<a href="https://colab.research.google.com/github/s4tlaw/KTH_AH2179/blob/main/Group%20Project/AH2179_XGBoostRegession.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
INTERVAL_MINUTES = 1
FORECAST_STEPS = 15

# PART I: Data Preprocessing

In [ ]:
import os
import numpy as np
import pandas as pd
from google.colab import drive

# =========================================================================
# CONFIGURATION: Set your desired resampling interval (1, 2, 5, etc.)
# =========================================================================
save_directory = '/content/drive/MyDrive/AH2179 GP5/Colab'
save_filename = f'processed_final_df_{INTERVAL_MINUTES}min.csv'
save_path = os.path.join(save_directory, save_filename)
raw_data_url = '/content/drive/MyDrive/AH2179 GP5/Colab/training_dataset.csv'

# =========================================================================
# GLOBAL HELPER FUNCTIONS (Must be defined outside the conditional blocks
# so they are always loaded and available in the current session namespace)
# =========================================================================
def mean_nonzero(speeds):
    """Computes the mean of non-zero speeds; returns 0.0 if empty or all-zero."""
    valid_speeds = speeds[speeds > 0]
    return valid_speeds.mean() if len(valid_speeds) > 0 else 0.0

# =========================================================================
# 1. MOUNT GOOGLE DRIVE
# =========================================================================
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive', force_remount=True)

# Ensure the target directory structure exists
os.makedirs(save_directory, exist_ok=True)

# =========================================================================
# 2. CHECK DRIVE FOR EXISTING PROCESSED FILE
# =========================================================================
if os.path.exists(save_path):
    print(f"Found pre-processed file on Google Drive: {save_path}")
    print("Loading directly to skip the raw data preprocessing pipeline...")
    final_df = pd.read_csv(save_path)
    final_df['DateTime'] = pd.to_datetime(final_df['DateTime'])
    print(f"Successfully loaded final_df ({len(final_df)} rows)!")
else:
    print(f"No pre-processed file found at: {save_path}")
    print("Initiating full data preprocessing pipeline from raw dataset...")

    # =========================================================================
    # 3. RUN FULL DATA PREPROCESSING PIPELINE (Only if file doesn't exist)
    # =========================================================================
    if not os.path.exists(raw_data_url):
        raise FileNotFoundError(f"Raw dataset missing! Please make sure it is at: {raw_data_url}")

    print(f"Loading raw training dataset from: {raw_data_url}...")
    df = pd.read_csv(raw_data_url, sep=';')

    # Clean and extract Portal IDs & map to sequence (1 to 8)
    print("Cleaning portals and mapping sequence sequences...")
    df['PORTAL_ID'] = df['PORTAL'].str.replace('E4S ', '').str.replace(',', '').astype(int)

    portal_sequence = {
        58140: 1,
        57820: 2,
        57435: 3,
        57055: 4,
        56780: 5,
        56490: 6,
        56160: 7,
        55620: 8
    }
    df['sequence'] = df['PORTAL_ID'].map(portal_sequence)

    # Convert to Datetime
    df['DateTime'] = pd.to_datetime(df['Date'].astype(str) + ' ' + df['Time'])

    # Combine sensor-level data per 1-minute interval
    print("Aggregating sensor data to 1-minute intervals...")
    portal_min_df = df.groupby(['sequence', 'DateTime']).agg(
        v=('SPEED_MS_AVG', mean_nonzero),
        f=('FLOW', 'sum')
    ).reset_index()

    # Resample into dynamic intervals based on user configuration
    print(f"Resampling and merging into custom {INTERVAL_MINUTES}-minute intervals...")
    portal_min_indexed = portal_min_df.set_index('DateTime')
    final_df = portal_min_indexed.groupby('sequence').resample(f'{INTERVAL_MINUTES}Min').agg({
        'v': mean_nonzero,
        'f': 'sum'
    }).reset_index()

    # Save interval-specific DataFrame to Google Drive for future fast-loads
    print(f"Saving processed DataFrame to: {save_path}...")
    final_df.to_csv(save_path, index=False)
    print(f"Successfully preprocessed and saved: {save_filename}!")

# Display sample records
display(final_df.head(10))

Found pre-processed file on Google Drive: /content/drive/MyDrive/AH2179 GP5/Colab/processed_final_df_1min.csv
Loading directly to skip the raw data preprocessing pipeline...
Successfully loaded final_df (2456648 rows)!


,sequence,DateTime,v,f
0,1,2021-06-01 04:00:00,20.830000,2.0
1,1,2021-06-01 04:01:00,22.313333,6.0
2,1,2021-06-01 04:02:00,21.945000,7.0
3,1,2021-06-01 04:03:00,24.025000,2.0
4,1,2021-06-01 04:04:00,25.276667,3.0
5,1,2021-06-01 04:05:00,21.666667,6.0
6,1,2021-06-01 04:06:00,22.640000,2.0
7,1,2021-06-01 04:07:00,22.780000,4.0
8,1,2021-06-01 04:08:00,24.585000,5.0
9,1,2021-06-01 04:09:00,21.390000,5.0


# PART II: $C_{max}$ estimator

In [ ]:
"""
Corridor Capacity (C_max) Model -- v8.2, Scoped to Portals 7 & 8 Only
====================================================================================
FIX vs v8.1: adds day_of_week / is_weekend features to the capacity
model's feature set. Diagnostic evidence for this change (see
portal78_diagnostic_extension.py output):

  - Check 1 (non-stationarity): C_max shows a statistically significant
    downward drift over the observed date range for both portals
    (Portal 7: Spearman r=-0.270, p=0.0001, -19.4% first-vs-last-20%;
     Portal 8: Spearman r=-0.332, p<0.0001, -20.5%). The v8.1 feature
    set has NO time-awareness at all, so it cannot account for this.
  - Check 3 (RF importances): f_trend_ratio_log + v_trend_delta combined
    contributed only ~3-4.5% importance for portals 7/8 -- consistent
    with these portals never reaching true v-f congestion inversion
    (confirmed separately), so the "approaching congestion" framing
    behind those two features doesn't apply as strongly here.

day_of_week is added (not a raw linear time-index) specifically to avoid
introducing look-ahead leakage risk into the shuffled K-Fold CV -- it is
cyclical/calendar-derived, not a monotonic trend proxy. A raw time-index
was considered and deliberately NOT included for this reason.

Everything else preserved from v8.1: same unified per-portal template
(identical feature set, identical search grids, no per-seq branching),
repeated K-Fold CV as the honest accuracy estimate, coverage-based day
filtering in the estimator (not any-NaN-drops-day).

Assumes `final_df` (columns: sequence, DateTime, v, f) is already loaded
in this session -- run your preprocessing cell first.
"""

import os
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb


os.makedirs('output', exist_ok=True)


# ============================================================================
# CONFIG
# ============================================================================

RAW_COLUMNS = dict(datetime='DateTime', sequence='sequence', speed='v', flow='f')

ACTIVE_PORTALS = [7, 8]   # SCOPE: only these two portals get a capacity model

AM_WINDOW_START = pd.to_datetime('07:30:00').time()
AM_WINDOW_END = pd.to_datetime('08:30:00').time()
MORNING_START = pd.to_datetime('04:00:00').time()
MORNING_END = pd.to_datetime('07:30:00').time()
_morning_start_dt = pd.to_datetime(MORNING_START.strftime('%H:%M:%S'))
_morning_end_dt = pd.to_datetime(MORNING_END.strftime('%H:%M:%S'))
MORNING_SPLIT = (_morning_start_dt + (_morning_end_dt - _morning_start_dt) / 2).time()

CAPACITY_N_SPLITS = 5
CAPACITY_N_REPEATS = 3
RANDOM_STATE = 42

CAPACITY_XGB_GRID = {
    'n_estimators': [40, 60, 100, 150],
    'max_depth': [2, 3, 4, 6],
    'learning_rate': [0.03, 0.07, 0.1],
}
CAPACITY_RF_GRID = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 4, 6, 8],
    'min_samples_leaf': [1, 3, 5, 10],
}
CAPACITY_SVR_GRID = {
    'C': [10, 50, 100, 500, 1000, 2000],
    'epsilon': [0.1, 0.2],
}

# v8.2: day_of_week + is_weekend added. Derived purely from the Date index
# already present in every portal's dataset -- no new data source, no
# look-ahead leakage (cyclical/calendar feature, not a monotonic trend).
CAPACITY_FEATURE_COLUMNS = [
    'v', 'f', 'c_early', 'f_trend_ratio_log', 'v_trend_delta',
    'day_of_week', 'is_weekend',
]

MIN_MORNING_WINDOW_COVERAGE = 0.5   # confirmed fix: coverage-based day filter,
                                     # NOT the old any-NaN-drops-day rule

EXPORT_DIR = '/content/drive/MyDrive/AH2179 GP5/prediction model v8.2 (portals 7-8, time-aware)'


# ============================================================================
# CAPACITY MODEL -- training dataset construction
# ============================================================================

def build_capacity_training_dataset(seq, raw_df):
    """One portal's capacity-model dataset: early-morning (04:00-07:30)
    features (v, f, c_early, trend ratios) + day-of-week/weekend features,
    and the C_max target from the 07:30-08:30 peak window (98th percentile
    of instantaneous C = v*f)."""
    p_df = raw_df[raw_df[RAW_COLUMNS['sequence']] == seq].copy()
    p_df['Date'] = p_df[RAW_COLUMNS['datetime']].dt.date
    p_df['C'] = p_df[RAW_COLUMNS['speed']] * p_df[RAW_COLUMNS['flow']]

    mask_early = (p_df[RAW_COLUMNS['datetime']].dt.time >= MORNING_START) & \
                 (p_df[RAW_COLUMNS['datetime']].dt.time < MORNING_END)
    early_data = p_df[mask_early]
    daily_v = early_data.groupby('Date')[RAW_COLUMNS['speed']].mean().rename('v')
    daily_f = early_data.groupby('Date')[RAW_COLUMNS['flow']].sum().rename('f')
    daily_c_early = early_data.groupby('Date')['C'].quantile(0.98).rename('c_early')

    mask_first_half = (p_df[RAW_COLUMNS['datetime']].dt.time >= MORNING_START) & \
                       (p_df[RAW_COLUMNS['datetime']].dt.time < MORNING_SPLIT)
    mask_second_half = (p_df[RAW_COLUMNS['datetime']].dt.time >= MORNING_SPLIT) & \
                        (p_df[RAW_COLUMNS['datetime']].dt.time < MORNING_END)
    f_first = p_df[mask_first_half].groupby('Date')[RAW_COLUMNS['flow']].sum().rename('f_first_half')
    f_second = p_df[mask_second_half].groupby('Date')[RAW_COLUMNS['flow']].sum().rename('f_second_half')
    f_trend_ratio_log = np.log1p(f_second / f_first.replace(0, np.nan)).rename('f_trend_ratio_log')
    v_first = p_df[mask_first_half].groupby('Date')[RAW_COLUMNS['speed']].mean().rename('v_first_half')
    v_second = p_df[mask_second_half].groupby('Date')[RAW_COLUMNS['speed']].mean().rename('v_second_half')
    v_trend_delta = (v_second - v_first).rename('v_trend_delta')

    features_df = pd.concat([daily_v, daily_f, daily_c_early, f_trend_ratio_log, v_trend_delta], axis=1)

    mask_target = (p_df[RAW_COLUMNS['datetime']].dt.time >= AM_WINDOW_START) & \
                  (p_df[RAW_COLUMNS['datetime']].dt.time <= AM_WINDOW_END)
    c_max_daily = p_df[mask_target].groupby('Date')['C'].quantile(0.98).rename('C_max')

    dataset = features_df.join(c_max_daily).dropna()
    dataset = dataset[dataset['C_max'] > 10]

    # v8.2: day-of-week / weekend features, derived from the Date index.
    date_index = pd.to_datetime(dataset.index)
    dataset['day_of_week'] = date_index.dayofweek
    dataset['is_weekend'] = (date_index.dayofweek >= 5).astype(int)

    return dataset


# ============================================================================
# CAPACITY MODEL -- stacked ensemble
# ============================================================================

def build_capacity_stack_search():
    """Fresh, untrained (RF, XGB, SVR) grid-search trio -- one instance per
    fold, so hyperparameters are re-tuned inside every outer fold rather
    than leaked from test data."""
    grid_xgb = GridSearchCV(
        xgb.XGBRegressor(objective='reg:absoluteerror', random_state=RANDOM_STATE),
        CAPACITY_XGB_GRID, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
    )
    grid_rf = GridSearchCV(
        RandomForestRegressor(random_state=RANDOM_STATE),
        CAPACITY_RF_GRID, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
    )
    grid_svr = GridSearchCV(SVR(), CAPACITY_SVR_GRID, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
    return grid_rf, grid_xgb, grid_svr


def fit_capacity_stack(X_scaled, y):
    grid_rf, grid_xgb, grid_svr = build_capacity_stack_search()
    grid_rf.fit(X_scaled, y)
    grid_xgb.fit(X_scaled, y)
    grid_svr.fit(X_scaled, y)
    stack = StackingRegressor(
        estimators=[('rf', grid_rf.best_estimator_), ('xgb', grid_xgb.best_estimator_), ('svr', grid_svr.best_estimator_)],
        final_estimator=RidgeCV(alphas=np.logspace(0, 5, 12)),
        passthrough=True, cv=5, n_jobs=-1,
    )
    model = TransformedTargetRegressor(regressor=stack, func=np.log1p, inverse_func=np.expm1)
    model.fit(X_scaled, y)
    return model


def train_and_evaluate_capacity_models(raw_df, portals):
    """Trains one FINAL capacity model per portal (on all data), restricted
    to `portals` ([7, 8]), AND reports a repeated-K-Fold CV estimate of
    honest out-of-sample accuracy. Also reports RF feature importances so
    the day_of_week/is_weekend additions can be checked directly."""
    capacity_models = {}
    cv_summary_rows, cv_fold_rows, importance_rows = [], [], []

    for seq in portals:
        print(f"\n[Capacity Model] Cross-validating portal {seq}...")
        dataset = build_capacity_training_dataset(seq, raw_df)
        X = dataset[CAPACITY_FEATURE_COLUMNS].values
        y = dataset['C_max'].values

        fold_r2, fold_mae, fold_rmse = [], [], []
        for repeat in range(CAPACITY_N_REPEATS):
            kf = KFold(n_splits=CAPACITY_N_SPLITS, shuffle=True, random_state=repeat)
            for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X)):
                scaler = RobustScaler()
                X_tr_scaled = scaler.fit_transform(X[train_idx])
                X_te_scaled = scaler.transform(X[test_idx])
                model = fit_capacity_stack(X_tr_scaled, y[train_idx])
                y_pred = model.predict(X_te_scaled)

                r2 = r2_score(y[test_idx], y_pred)
                mae = mean_absolute_error(y[test_idx], y_pred)
                rmse = np.sqrt(mean_squared_error(y[test_idx], y_pred))
                fold_r2.append(r2); fold_mae.append(mae); fold_rmse.append(rmse)
                cv_fold_rows.append({'Portal': seq, 'Repeat': repeat, 'Fold': fold_idx,
                                      'R2': round(r2, 3), 'MAE': round(mae, 3), 'RMSE': round(rmse, 3)})

        scaler_full = RobustScaler()
        X_full_scaled = scaler_full.fit_transform(X)
        final_model = fit_capacity_stack(X_full_scaled, y)
        capacity_models[seq] = {'model': final_model, 'scaler': scaler_full}

        rf_estimator = dict(final_model.regressor_.named_estimators_)['rf']
        importances = dict(zip(CAPACITY_FEATURE_COLUMNS, rf_estimator.feature_importances_.round(3)))
        importance_rows.append({'Portal': seq, **importances})

        cv_summary_rows.append({
            'Portal': seq, 'n_folds_evaluated': len(fold_r2),
            'R2_mean': round(np.mean(fold_r2), 3), 'R2_std': round(np.std(fold_r2), 3),
            'R2_min': round(np.min(fold_r2), 3), 'R2_max': round(np.max(fold_r2), 3),
            'MAE_mean': round(np.mean(fold_mae), 3), 'RMSE_mean': round(np.mean(fold_rmse), 3),
        })

    return capacity_models, pd.DataFrame(cv_summary_rows), pd.DataFrame(cv_fold_rows), pd.DataFrame(importance_rows)


# ============================================================================
# CAPACITY ESTIMATOR -- applies an ALREADY-TRAINED capacity model to new
# data. Coverage-based day filtering (not any-NaN-drops-day). v8.2 adds
# day_of_week / is_weekend to match the training feature set.
# ============================================================================

def estimate_daily_capacity(seq, raw_df, capacity_models, interval_minutes=1, verbose=True):
    """Applies an ALREADY-TRAINED capacity model to new data to produce a
    daily C_max_pred series for one portal. `seq` must be a key in
    `capacity_models` -- with this scoped version, only 7 and 8 qualify."""
    if seq not in capacity_models:
        raise KeyError(
            f"No capacity model found for portal/sequence {seq}. This v8.2 build only "
            f"trains portals {ACTIVE_PORTALS}."
        )

    p_df = raw_df[raw_df[RAW_COLUMNS['sequence']] == seq].copy()
    p_df['Date'] = p_df[RAW_COLUMNS['datetime']].dt.date
    p_df['Time'] = p_df[RAW_COLUMNS['datetime']].dt.time
    p_df['C'] = p_df[RAW_COLUMNS['speed']] * p_df[RAW_COLUMNS['flow']]

    mask_morning_all = (p_df['Time'] >= MORNING_START) & (p_df['Time'] < MORNING_END)
    morning_rows_per_day = p_df[mask_morning_all].groupby('Date').size()
    expected_morning_rows = int(
        (pd.to_datetime(MORNING_END.strftime('%H:%M:%S')) -
         pd.to_datetime(MORNING_START.strftime('%H:%M:%S'))).total_seconds() / 60.0 // interval_minutes
    ) + 1

    all_days_seen = set(p_df['Date'].unique())
    covered_days = set(morning_rows_per_day[
        morning_rows_per_day >= MIN_MORNING_WINDOW_COVERAGE * expected_morning_rows].index)
    dropped_for_coverage = all_days_seen - covered_days
    if verbose and dropped_for_coverage:
        print(f"  [Capacity Estimator, portal {seq}] Dropped {len(dropped_for_coverage)} day(s) "
              f"for insufficient morning coverage (<{MIN_MORNING_WINDOW_COVERAGE:.0%} of "
              f"~{expected_morning_rows} expected rows).")

    p_df = p_df[p_df['Date'].isin(covered_days)]
    mask_early = (p_df['Time'] >= MORNING_START) & (p_df['Time'] < MORNING_END)
    early_data = p_df[mask_early]
    daily_v = early_data.groupby('Date')[RAW_COLUMNS['speed']].mean().rename('v')
    daily_f = early_data.groupby('Date')[RAW_COLUMNS['flow']].sum().rename('f')
    daily_c_early = early_data.groupby('Date')['C'].quantile(0.98).rename('c_early')

    mask_fh = (p_df['Time'] >= MORNING_START) & (p_df['Time'] < MORNING_SPLIT)
    mask_sh = (p_df['Time'] >= MORNING_SPLIT) & (p_df['Time'] < MORNING_END)
    f_first = p_df[mask_fh].groupby('Date')[RAW_COLUMNS['flow']].sum().rename('f_first_half')
    f_second = p_df[mask_sh].groupby('Date')[RAW_COLUMNS['flow']].sum().rename('f_second_half')
    f_trend_df = pd.concat([f_first, f_second], axis=1).fillna(0.0)
    EPS = 1.0   # robust to zero-flow half-windows -- avoids the 0/0 -> NaN ->
                # whole-day-dropped failure mode from the original estimator
    ratio = (f_trend_df['f_second_half'] + EPS) / (f_trend_df['f_first_half'] + EPS)
    f_trend_ratio_log = np.log1p(ratio - 1.0).rename('f_trend_ratio_log')

    v_first = p_df[mask_fh].groupby('Date')[RAW_COLUMNS['speed']].mean().rename('v_first_half')
    v_second = p_df[mask_sh].groupby('Date')[RAW_COLUMNS['speed']].mean().rename('v_second_half')
    v_trend_delta = (v_second.fillna(0.0) - v_first.fillna(0.0)).rename('v_trend_delta')

    features_df = pd.concat([daily_v, daily_f, daily_c_early, f_trend_ratio_log, v_trend_delta], axis=1
                             ).reset_index().rename(columns={'index': 'Date'}).dropna()

    # v8.2: day_of_week / is_weekend, matching the training feature set.
    date_index = pd.to_datetime(features_df['Date'])
    features_df['day_of_week'] = date_index.dt.dayofweek
    features_df['is_weekend'] = (date_index.dt.dayofweek >= 5).astype(int)

    entry = capacity_models[seq]
    X = features_df[CAPACITY_FEATURE_COLUMNS].values
    features_df['C_max_pred'] = entry['model'].predict(entry['scaler'].transform(X))

    if verbose:
        print(f"  [Capacity Estimator, portal {seq}] {len(features_df)} of "
              f"{len(all_days_seen)} day(s) produced a valid C_max_pred.")
    return features_df[['Date', 'C_max_pred']]


def detect_portals(raw_df, restrict_to=None):
    portals = sorted(int(s) for s in raw_df[RAW_COLUMNS['sequence']].unique())
    if restrict_to is not None:
        missing = [p for p in restrict_to if p not in portals]
        if missing:
            raise ValueError(f"Requested portals {missing} not found in data. Detected: {portals}")
        return sorted(restrict_to)
    return portals


# ============================================================================
# MAIN
# ============================================================================
if __name__ == '__main__':
    if 'final_df' not in globals():
        raise NameError(
            "final_df not found in this session. Run your preprocessing cell first -- "
            "it produces `final_df` (either by loading the cached "
            "processed_final_df_{N}min.csv from Drive, or by building it fresh from "
            "training_dataset.csv)."
        )

    raw_df = final_df.rename(columns={
        'DateTime': RAW_COLUMNS['datetime'], 'sequence': RAW_COLUMNS['sequence'],
        'v': RAW_COLUMNS['speed'], 'f': RAW_COLUMNS['flow']
    })
    portals_detected = detect_portals(raw_df, restrict_to=ACTIVE_PORTALS)
    print(f"[v8.2 Part 1] Scoped to portal(s): {portals_detected} (portals 1-6 skipped -- "
          f"reduced training scope). Feature set now includes day_of_week/is_weekend.")

    capacity_models, capacity_cv_summary, capacity_cv_folds, capacity_importances = \
        train_and_evaluate_capacity_models(raw_df, portals_detected)

    print("\n=================== CAPACITY MODEL CV RESULTS (portals 7-8, v8.2) ===================")
    print(capacity_cv_summary.to_string(index=False))
    print("\n--- RF feature importances (check day_of_week/is_weekend contribution) ---")
    print(capacity_importances.to_string(index=False))

    capacity_cv_folds.to_csv('output/capacity_model_cv_folds_p78_v82.csv', index=False)
    print("\nSaved fold-level detail: output/capacity_model_cv_folds_p78_v82.csv")

    os.makedirs(EXPORT_DIR, exist_ok=True)
    capacity_artifacts = {
        'portal_models': capacity_models,
        'feature_order': CAPACITY_FEATURE_COLUMNS,
        'active_portals': ACTIVE_PORTALS,
        'description': (
            "Per-portal stacking regressor (RF + XGB + SVR, RidgeCV meta-learner, "
            "log1p/expm1 target transform) predicting daily peak carrying capacity "
            "(C_max) from early-morning (04:00-07:30) features plus day_of_week/"
            "is_weekend. v8.2: added time-awareness after confirming C_max shows "
            "significant date-range drift for portals 7/8 (non-stationary target). "
            "SCOPED to portals 7 and 8 only. Built from `final_df`."
        ),
    }
    export_path = os.path.join(EXPORT_DIR, 'cmax_prediction_model_p78_v82.joblib')
    joblib.dump(capacity_artifacts, export_path)
    print(f"\n[Export] Saved capacity model artifact to: {export_path}")
    print("\n[v8.2 Part 1] Complete.")

[v8.2 Part 1] Scoped to portal(s): [7, 8] (portals 1-6 skipped -- reduced training scope). Feature set now includes day_of_week/is_weekend.

[Capacity Model] Cross-validating portal 7...

[Capacity Model] Cross-validating portal 8...

=================== CAPACITY MODEL CV RESULTS (portals 7-8, v8.2) ===================
 Portal  n_folds_evaluated  R2_mean  R2_std  R2_min  R2_max  MAE_mean  RMSE_mean
      7                 15    0.836   0.067   0.690   0.943    68.586     87.798
      8                 15    0.816   0.082   0.611   0.916    65.414     83.940

--- RF feature importances (check day_of_week/is_weekend contribution) ---
 Portal     v     f  c_early  f_trend_ratio_log  v_trend_delta  day_of_week  is_weekend
      7 0.022 0.554    0.373              0.024          0.021        0.004       0.001
      8 0.045 0.390    0.520              0.025          0.014        0.005       0.000

Saved fold-level detail: output/capacity_model_cv_folds_p78_v82.csv

[Export] Saved capacity mo

## $C_{max}$ evaluation metrics

In [ ]:
"""
C_max Model v8.2 -- Diagnostic and Metrics Evaluation
================================================================================
Run AFTER the v8.2 capacity-model script in the SAME session.

Required objects already created by v8.2:
    - raw_df
    - capacity_models
    - build_capacity_training_dataset()
    - fit_capacity_stack()
    - CAPACITY_FEATURE_COLUMNS
    - ACTIVE_PORTALS

Scope:
    - Portals 7 and 8
    - Same repeated 5-fold nested CV configuration as v8.2
    - Captures out-of-fold predictions and detailed diagnostics

Important:
    This script re-runs the nested CV process so it can obtain honest
    out-of-fold predictions and residuals. It is therefore computationally
    expensive, similarly to the v8.2 training process.

Outputs:
    output/v82_diagnostic/
        - v82_fold_metrics.csv
        - v82_oof_predictions.csv
        - v82_summary_metrics.csv
        - v82_feature_target_correlations.csv
        - v82_residual_feature_correlations.csv
        - v82_segment_metrics.csv
        - v82_final_rf_importances.csv
        - v82_high_error_days.csv
"""

import os
import numpy as np
import pandas as pd

from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# =============================================================================
# CONFIGURATION
# =============================================================================

DIAGNOSTIC_PORTALS = [7, 8]

# Must match v8.2 CV configuration exactly.
N_SPLITS = CAPACITY_N_SPLITS
N_REPEATS = CAPACITY_N_REPEATS

DIAGNOSTIC_DIR = "output/v82_diagnostic"
os.makedirs(DIAGNOSTIC_DIR, exist_ok=True)


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def safe_correlation(x, y):
    """
    Return Pearson and Spearman correlations safely.
    If there is too little variation or too few rows, return NaN values.
    """
    temp = pd.DataFrame({"x": x, "y": y}).dropna()

    if len(temp) < 3:
        return np.nan, np.nan, np.nan, np.nan

    if temp["x"].nunique() < 2 or temp["y"].nunique() < 2:
        return np.nan, np.nan, np.nan, np.nan

    pearson_r, pearson_p = pearsonr(temp["x"], temp["y"])
    spearman_r, spearman_p = spearmanr(temp["x"], temp["y"])

    return pearson_r, pearson_p, spearman_r, spearman_p


def metric_row(actual, prediction):
    """
    Calculate a consistent set of regression metrics.
    """
    residual = prediction - actual
    abs_error = np.abs(residual)

    return {
        "n_predictions": len(actual),
        "R2": r2_score(actual, prediction),
        "MAE": mean_absolute_error(actual, prediction),
        "RMSE": np.sqrt(mean_squared_error(actual, prediction)),
        "Bias_mean_error": np.mean(residual),
        "Median_error": np.median(residual),
        "Median_absolute_error": np.median(abs_error),
        "P90_absolute_error": np.quantile(abs_error, 0.90),
        "Max_absolute_error": np.max(abs_error),
    }


def label_error_band(abs_error, q50, q90):
    """
    Human-readable error band for diagnostic tables.
    """
    if abs_error <= q50:
        return "Low error"
    if abs_error <= q90:
        return "Moderate error"
    return "High error"


# =============================================================================
# PRE-FLIGHT CHECKS
# =============================================================================

required_objects = [
    "raw_df",
    "capacity_models",
    "build_capacity_training_dataset",
    "fit_capacity_stack",
    "CAPACITY_FEATURE_COLUMNS",
]

missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise NameError(
        "The v8.2 model script must run first in the same notebook session. "
        f"Missing objects: {missing_objects}"
    )

missing_models = [seq for seq in DIAGNOSTIC_PORTALS if seq not in capacity_models]

if missing_models:
    raise KeyError(
        f"Models for portals {missing_models} were not found in capacity_models. "
        "Confirm that v8.2 completed successfully."
    )

print("=" * 80)
print("V8.2 C_max DIAGNOSTIC AND METRICS EVALUATION")
print("=" * 80)
print(f"Portals evaluated: {DIAGNOSTIC_PORTALS}")
print(f"Outer evaluation: {N_SPLITS}-fold CV repeated {N_REPEATS} times")
print(f"Total out-of-fold evaluations per portal: {N_SPLITS * N_REPEATS}")
print(f"Feature set: {CAPACITY_FEATURE_COLUMNS}")


# =============================================================================
# STEP 1: REPEATED NESTED CV WITH OUT-OF-FOLD PREDICTION CAPTURE
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1: GENERATING REPEATED OUT-OF-FOLD PREDICTIONS")
print("=" * 80)

fold_metric_rows = []
oof_prediction_rows = []

for seq in DIAGNOSTIC_PORTALS:
    print(f"\n[Portal {seq}] Building daily dataset...")

    dataset = build_capacity_training_dataset(seq, raw_df).copy()
    dataset.index = pd.to_datetime(dataset.index)

    X = dataset[CAPACITY_FEATURE_COLUMNS].values
    y = dataset["C_max"].values

    print(f"[Portal {seq}] Usable daily observations: {len(dataset)}")

    for repeat in range(N_REPEATS):
        kf = KFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=repeat,
        )

        for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
            print(
                f"  Portal {seq} | repeat {repeat + 1}/{N_REPEATS} "
                f"| fold {fold + 1}/{N_SPLITS}"
            )

            scaler = RobustScaler()
            X_train_scaled = scaler.fit_transform(X[train_idx])
            X_test_scaled = scaler.transform(X[test_idx])

            # Uses v8.2's nested GridSearchCV + stack construction.
            model = fit_capacity_stack(
                X_train_scaled,
                y[train_idx],
            )

            y_pred = model.predict(X_test_scaled)

            fold_result = metric_row(
                actual=y[test_idx],
                prediction=y_pred,
            )

            fold_metric_rows.append({
                "Portal": seq,
                "Repeat": repeat,
                "Fold": fold,
                "Train_n": len(train_idx),
                "Test_n": len(test_idx),
                **{
                    key: round(value, 6)
                    for key, value in fold_result.items()
                },
            })

            test_dates = dataset.index[test_idx]

            for local_position, row_position in enumerate(test_idx):
                actual = y[row_position]
                prediction = y_pred[local_position]
                residual = prediction - actual

                row = {
                    "Portal": seq,
                    "Repeat": repeat,
                    "Fold": fold,
                    "Date": test_dates[local_position],
                    "C_max_actual": actual,
                    "C_max_pred": prediction,
                    "Residual_pred_minus_actual": residual,
                    "Absolute_error": abs(residual),
                    "Squared_error": residual ** 2,
                }

                # Store every feature for residual diagnostics.
                for feature in CAPACITY_FEATURE_COLUMNS:
                    row[feature] = dataset.iloc[row_position][feature]

                row["day_name"] = test_dates[local_position].day_name()
                row["is_weekend_label"] = (
                    "Weekend"
                    if test_dates[local_position].dayofweek >= 5
                    else "Weekday"
                )

                oof_prediction_rows.append(row)


fold_metrics_df = pd.DataFrame(fold_metric_rows)
oof_predictions_df = pd.DataFrame(oof_prediction_rows)

fold_metrics_df.to_csv(
    os.path.join(DIAGNOSTIC_DIR, "v82_fold_metrics.csv"),
    index=False,
)

oof_predictions_df.to_csv(
    os.path.join(DIAGNOSTIC_DIR, "v82_oof_predictions.csv"),
    index=False,
)

print("\nSaved:")
print(f"  {DIAGNOSTIC_DIR}/v82_fold_metrics.csv")
print(f"  {DIAGNOSTIC_DIR}/v82_oof_predictions.csv")


# =============================================================================
# STEP 2: FOLD-STABILITY METRICS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2: FOLD-STABILITY METRICS")
print("=" * 80)

summary_rows = []

for seq in DIAGNOSTIC_PORTALS:
    portal_fold_metrics = fold_metrics_df[fold_metrics_df["Portal"] == seq]
    portal_oof = oof_predictions_df[oof_predictions_df["Portal"] == seq]

    pooled_metrics = metric_row(
        actual=portal_oof["C_max_actual"].values,
        prediction=portal_oof["C_max_pred"].values,
    )

    summary_rows.append({
        "Portal": seq,
        "n_folds_evaluated": len(portal_fold_metrics),
        "n_repeated_oof_predictions": len(portal_oof),
        "Fold_R2_mean": portal_fold_metrics["R2"].mean(),
        "Fold_R2_std": portal_fold_metrics["R2"].std(ddof=0),
        "Fold_R2_min": portal_fold_metrics["R2"].min(),
        "Fold_R2_max": portal_fold_metrics["R2"].max(),
        "Fold_MAE_mean": portal_fold_metrics["MAE"].mean(),
        "Fold_MAE_std": portal_fold_metrics["MAE"].std(ddof=0),
        "Fold_RMSE_mean": portal_fold_metrics["RMSE"].mean(),
        "Fold_RMSE_std": portal_fold_metrics["RMSE"].std(ddof=0),
        "Pooled_OOF_R2": pooled_metrics["R2"],
        "Pooled_OOF_MAE": pooled_metrics["MAE"],
        "Pooled_OOF_RMSE": pooled_metrics["RMSE"],
        "Pooled_OOF_Bias": pooled_metrics["Bias_mean_error"],
        "Pooled_OOF_Median_absolute_error": pooled_metrics[
            "Median_absolute_error"
        ],
        "Pooled_OOF_P90_absolute_error": pooled_metrics[
            "P90_absolute_error"
        ],
        "Pooled_OOF_Max_absolute_error": pooled_metrics[
            "Max_absolute_error"
        ],
    })

summary_metrics_df = pd.DataFrame(summary_rows)

for column in summary_metrics_df.columns:
    if column not in ["Portal", "n_folds_evaluated", "n_repeated_oof_predictions"]:
        summary_metrics_df[column] = summary_metrics_df[column].round(3)

print(summary_metrics_df.to_string(index=False))

summary_metrics_df.to_csv(
    os.path.join(DIAGNOSTIC_DIR, "v82_summary_metrics.csv"),
    index=False,
)


# =============================================================================
# STEP 3: FEATURE-TARGET CORRELATION
# =============================================================================

print("\n" + "=" * 80)
print("STEP 3: RAW FEATURE -> C_max RELATIONSHIP")
print("=" * 80)

feature_target_rows = []

for seq in DIAGNOSTIC_PORTALS:
    dataset = build_capacity_training_dataset(seq, raw_df).copy()

    for feature in CAPACITY_FEATURE_COLUMNS:
        pearson_r, pearson_p, spearman_r, spearman_p = safe_correlation(
            dataset[feature],
            dataset["C_max"],
        )

        feature_target_rows.append({
            "Portal": seq,
            "Feature": feature,
            "Pearson_r": pearson_r,
            "Pearson_p": pearson_p,
            "Spearman_r": spearman_r,
            "Spearman_p": spearman_p,
        })

feature_target_corr_df = pd.DataFrame(feature_target_rows)

for column in ["Pearson_r", "Pearson_p", "Spearman_r", "Spearman_p"]:
    feature_target_corr_df[column] = feature_target_corr_df[column].round(4)

print(feature_target_corr_df.to_string(index=False))

feature_target_corr_df.to_csv(
    os.path.join(DIAGNOSTIC_DIR, "v82_feature_target_correlations.csv"),
    index=False,
)


# =============================================================================
# STEP 4: RESIDUAL DIAGNOSTIC
# =============================================================================

print("\n" + "=" * 80)
print("STEP 4: RESIDUAL -> FEATURE CORRELATION")
print("=" * 80)
print(
    "Purpose: identifies whether any feature remains systematically related "
    "to prediction error. A meaningful residual correlation suggests that "
    "the model may not be capturing that feature's relationship fully."
)

residual_feature_rows = []

for seq in DIAGNOSTIC_PORTALS:
    portal_oof = oof_predictions_df[
        oof_predictions_df["Portal"] == seq
    ].copy()

    for feature in CAPACITY_FEATURE_COLUMNS:
        pearson_r, pearson_p, spearman_r, spearman_p = safe_correlation(
            portal_oof[feature],
            portal_oof["Residual_pred_minus_actual"],
        )

        residual_feature_rows.append({
            "Portal": seq,
            "Feature": feature,
            "Pearson_r_with_residual": pearson_r,
            "Pearson_p": pearson_p,
            "Spearman_r_with_residual": spearman_r,
            "Spearman_p": spearman_p,
        })

residual_feature_corr_df = pd.DataFrame(residual_feature_rows)

for column in [
    "Pearson_r_with_residual",
    "Pearson_p",
    "Spearman_r_with_residual",
    "Spearman_p",
]:
    residual_feature_corr_df[column] = residual_feature_corr_df[column].round(4)

print(residual_feature_corr_df.to_string(index=False))

residual_feature_corr_df.to_csv(
    os.path.join(DIAGNOSTIC_DIR, "v82_residual_feature_correlations.csv"),
    index=False,
)


# =============================================================================
# STEP 5: WEEKDAY / WEEKEND / DAY-OF-WEEK PERFORMANCE
# =============================================================================

print("\n" + "=" * 80)
print("STEP 5: CALENDAR-SEGMENT PERFORMANCE")
print("=" * 80)
print(
    "Purpose: tests whether the newly added calendar features improve or "
    "fail systematically on particular days."
)

segment_metric_rows = []

for seq in DIAGNOSTIC_PORTALS:
    portal_oof = oof_predictions_df[
        oof_predictions_df["Portal"] == seq
    ].copy()

    # Weekday versus weekend.
    for segment_type, segment_df in portal_oof.groupby("is_weekend_label"):
        if len(segment_df) < 3:
            continue

        metrics = metric_row(
            actual=segment_df["C_max_actual"].values,
            prediction=segment_df["C_max_pred"].values,
        )

        segment_metric_rows.append({
            "Portal": seq,
            "Segment_type": "Weekday_vs_weekend",
            "Segment": segment_type,
            **metrics,
        })

    # Individual day of week.
    for day_name, segment_df in portal_oof.groupby("day_name"):
        if len(segment_df) < 3:
            continue

        metrics = metric_row(
            actual=segment_df["C_max_actual"].values,
            prediction=segment_df["C_max_pred"].values,
        )

        segment_metric_rows.append({
            "Portal": seq,
            "Segment_type": "Day_of_week",
            "Segment": day_name,
            **metrics,
        })

segment_metrics_df = pd.DataFrame(segment_metric_rows)

numeric_metric_columns = [
    "R2",
    "MAE",
    "RMSE",
    "Bias_mean_error",
    "Median_error",
    "Median_absolute_error",
    "P90_absolute_error",
    "Max_absolute_error",
]

for column in numeric_metric_columns:
    if column in segment_metrics_df.columns:
        segment_metrics_df[column] = segment_metrics_df[column].round(3)

print(segment_metrics_df.to_string(index=False))

segment_metrics_df.to_csv(
    os.path.join(DIAGNOSTIC_DIR, "v82_segment_metrics.csv"),
    index=False,
)


# =============================================================================
# STEP 6: IDENTIFY THE MOST DIFFICULT DAYS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 6: HIGH-ERROR DAY DIAGNOSTIC")
print("=" * 80)
print(
    "Each calendar day appears once per repeat. The table below aggregates "
    "error across repeats to identify days that remain systematically hard."
)

high_error_rows = []

for seq in DIAGNOSTIC_PORTALS:
    portal_oof = oof_predictions_df[
        oof_predictions_df["Portal"] == seq
    ].copy()

    daily_error = (
        portal_oof
        .groupby(["Portal", "Date", "day_name", "is_weekend_label"], as_index=False)
        .agg(
            C_max_actual=("C_max_actual", "mean"),
            C_max_pred_mean=("C_max_pred", "mean"),
            Residual_mean=("Residual_pred_minus_actual", "mean"),
            Absolute_error_mean=("Absolute_error", "mean"),
            Absolute_error_max=("Absolute_error", "max"),
            n_oof_appearances=("Absolute_error", "size"),
        )
    )

    q50 = daily_error["Absolute_error_mean"].quantile(0.50)
    q90 = daily_error["Absolute_error_mean"].quantile(0.90)

    daily_error["Error_band"] = daily_error["Absolute_error_mean"].apply(
        lambda value: label_error_band(value, q50, q90)
    )

    high_error_rows.append(daily_error)

high_error_days_df = pd.concat(high_error_rows, ignore_index=True)

high_error_days_df = high_error_days_df.sort_values(
    ["Portal", "Absolute_error_mean"],
    ascending=[True, False],
)

for column in [
    "C_max_actual",
    "C_max_pred_mean",
    "Residual_mean",
    "Absolute_error_mean",
    "Absolute_error_max",
]:
    high_error_days_df[column] = high_error_days_df[column].round(3)

print("\nTop 10 difficult days per portal:")
print(
    high_error_days_df
    .groupby("Portal", group_keys=False)
    .head(10)
    .to_string(index=False)
)

high_error_days_df.to_csv(
    os.path.join(DIAGNOSTIC_DIR, "v82_high_error_days.csv"),
    index=False,
)


# =============================================================================
# STEP 7: FINAL-MODEL RF FEATURE IMPORTANCE
# =============================================================================

print("\n" + "=" * 80)
print("STEP 7: FINAL RF FEATURE IMPORTANCE")
print("=" * 80)
print(
    "These are descriptive importances from the RF base learner in the "
    "final full-data stack. They do not prove causality."
)

importance_rows = []

for seq in DIAGNOSTIC_PORTALS:
    final_ttr_model = capacity_models[seq]["model"]
    final_stack = final_ttr_model.regressor_

    rf_estimator = dict(final_stack.named_estimators_)["rf"]

    importances = dict(
        zip(
            CAPACITY_FEATURE_COLUMNS,
            rf_estimator.feature_importances_.round(4),
        )
    )

    importance_rows.append({
        "Portal": seq,
        **importances,
    })

final_rf_importances_df = pd.DataFrame(importance_rows)

print(final_rf_importances_df.to_string(index=False))

final_rf_importances_df.to_csv(
    os.path.join(DIAGNOSTIC_DIR, "v82_final_rf_importances.csv"),
    index=False,
)


# =============================================================================
# STEP 8: FEATURE REDUNDANCY / COLLINEARITY CHECK
# =============================================================================

print("\n" + "=" * 80)
print("STEP 8: FEATURE REDUNDANCY CHECK")
print("=" * 80)
print(
    "Focus: confirms whether f and c_early remain highly collinear after "
    "adding calendar features. High collinearity is not necessarily harmful "
    "to RF/XGB, but it means their importances should not be interpreted "
    "as fully independent signals."
)

for seq in DIAGNOSTIC_PORTALS:
    dataset = build_capacity_training_dataset(seq, raw_df).copy()

    pearson_r, pearson_p, spearman_r, spearman_p = safe_correlation(
        dataset["f"],
        dataset["c_early"],
    )

    print(
        f"\nPortal {seq}: f versus c_early\n"
        f"  Pearson r  = {pearson_r:.4f}, p = {pearson_p:.6f}\n"
        f"  Spearman r = {spearman_r:.4f}, p = {spearman_p:.6f}"
    )


# =============================================================================
# FINAL INTERPRETATION GUIDE
# =============================================================================

print("\n" + "=" * 80)
print("INTERPRETATION GUIDE")
print("=" * 80)
print(
    """
1. Fold_R2_mean and Fold_R2_std
   - Primary generalization estimate.
   - Higher mean is better; lower standard deviation means more stable
     performance across different held-out groups of days.

2. Pooled_OOF metrics
   - Calculated over all repeated out-of-fold predictions together.
   - Use MAE/RMSE to express prediction error in original C_max units.
   - Bias near zero means the model does not systematically overpredict
     or underpredict C_max overall.

3. Residual-feature correlations
   - Values near zero suggest the model has captured that feature's
     association reasonably well.
   - A material correlation, especially |r| >= 0.20 with a small p-value,
     suggests systematic under/overprediction in part of feature space.

4. Calendar-segment metrics
   - Compare Weekday versus Weekend and individual weekday rows.
   - A large difference in bias or MAE would indicate that the calendar
     features may still not represent the weekly effect sufficiently.

5. High-error days
   - If the same dates appear repeatedly as difficult, inspect their raw
     data coverage, unusual flow/speed profile, sensor quality, incident
     status, or whether they are holidays/special events.

6. RF feature importance
   - v8.2 day_of_week/is_weekend importance near zero means calendar
     features are not strongly used by RF.
   - Do not remove them solely from one result: check whether CV mean/std,
     residual patterns, and weekday/weekend metrics jointly support removal.

All CSV outputs saved in:
    """
)
print(DIAGNOSTIC_DIR)
print("\n=================== V8.2 DIAGNOSTIC COMPLETE ===================")

V8.2 C_max DIAGNOSTIC AND METRICS EVALUATION
Portals evaluated: [7, 8]
Outer evaluation: 5-fold CV repeated 3 times
Total out-of-fold evaluations per portal: 15
Feature set: ['v', 'f', 'c_early', 'f_trend_ratio_log', 'v_trend_delta', 'day_of_week', 'is_weekend']

STEP 1: GENERATING REPEATED OUT-OF-FOLD PREDICTIONS

[Portal 7] Building daily dataset...
[Portal 7] Usable daily observations: 204
  Portal 7 | repeat 1/3 | fold 1/5
  Portal 7 | repeat 1/3 | fold 2/5
  Portal 7 | repeat 1/3 | fold 3/5
  Portal 7 | repeat 1/3 | fold 4/5
  Portal 7 | repeat 1/3 | fold 5/5
  Portal 7 | repeat 2/3 | fold 1/5
  Portal 7 | repeat 2/3 | fold 2/5
  Portal 7 | repeat 2/3 | fold 3/5
  Portal 7 | repeat 2/3 | fold 4/5
  Portal 7 | repeat 2/3 | fold 5/5
  Portal 7 | repeat 3/3 | fold 1/5
  Portal 7 | repeat 3/3 | fold 2/5
  Portal 7 | repeat 3/3 | fold 3/5
  Portal 7 | repeat 3/3 | fold 4/5
  Portal 7 | repeat 3/3 | fold 5/5

[Portal 8] Building daily dataset...
[Portal 8] Usable daily observations: 204

# PART III: $f$ prediction model for portal 8

In [ ]:
"""
Corridor f/S_pred Forecaster -- FINAL, compatible with C_max Model v8.2
================================================================================
Retrains the finalized Part 2 forecaster using the NEW C_max v8.2 model.

Why retraining is required
--------------------------
C_max v8.2 adds calendar features to its daily capacity estimate:
    ['v', 'f', 'c_early', 'f_trend_ratio_log', 'v_trend_delta',
     'day_of_week', 'is_weekend']

That changes C_max_pred and therefore changes:
    S_pred = (v * f) / C_max_pred

The old Part 2 artifact must NOT simply be paired with the new C_max model:
it was trained on S_pred calculated under the old capacity model. This script
rebuilds S_pred with C_max v8.2 and retrains Part 2 end-to-end.

Part 2 input scope remains unchanged:
    - Portals 7 and 8 only
    - Input variables: [v, f, S_pred] for each portal
    - Current value + lag 1 + lag 2
    - 18 features total
    - No additional Part 2 forecaster inputs

Locked per-target settings, selected from the previous v8.2 Part 2 ablation:
    f       : 5-minute centered target smoothing, level prediction
              smooth_window=5, predict_delta=False
    S_pred  : 5-minute centered target smoothing, delta prediction
              smooth_window=5, predict_delta=True

C_max v8.2 additions are used ONLY while calculating daily C_max_pred:
    day_of_week = Monday=0, ..., Sunday=6
    is_weekend  = 1 for Saturday/Sunday, else 0

Prerequisites
-------------
1. `final_df` exists in the current session, with columns:
       sequence, DateTime, v, f
2. New C_max v8.2 artifact has been saved to Drive or output/.
   Update CMAX_MODEL_CANDIDATE_PATHS below only if you used a custom name.

Outputs
-------
    output/cv_fold_detail_log_p78_FINAL_cmax_v82.csv
    output/fS_prediction_model_p78_FINAL_cmax_v82.joblib   (fallback if Drive unavailable)

Drive export target
-------------------
    /content/drive/MyDrive/AH2179 GP5/prediction model v8.1 (portals 7-8)/
        fS_prediction_model_p78_FINAL_cmax_v82.joblib
"""

import os
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb

os.makedirs('output', exist_ok=True)

# =============================================================================
# CONFIGURATION
# =============================================================================

RAW_COLUMNS = {'datetime': 'DateTime', 'sequence': 'sequence', 'speed': 'v', 'flow': 'f'}
ACTIVE_PORTALS = [7, 8]
TARGET_PORTAL_SEQ = 8

INTERVAL_MINUTES = 1
FORECAST_STEPS = 15
HORIZON_MINUTES = list(range(1, FORECAST_STEPS + 1))
ACTIVE_FORECAST_TARGETS = ['f', 'S_pred']
BASE_VARS = ['v', 'f', 'S_pred']
LAG_STEP_LIST = [1, 2]
MAX_LAG_NEEDED = max(LAG_STEP_LIST)

# Final per-target settings. Do not replace these with one global setting.
TARGET_TRAINING_CONFIG = {
    'f': {
        'smooth_window': 5,
        'predict_delta': False,
        'documented_reliable_through_min': 15,
    },
    'S_pred': {
        'smooth_window': 5,
        'predict_delta': True,
        'documented_reliable_through_min': 8,
    },
}

N_CV_FOLDS = 5
N_CV_REPEATS = 3
SEARCH_CV_FOLDS = 5
N_SEARCH_ITER = 20
EARLY_STOPPING_ROUNDS = 15
MAX_N_ESTIMATORS_WITH_EARLY_STOP = 500
RANDOM_STATE = 42

AM_WINDOW_START = pd.to_datetime('07:30:00').time()
AM_WINDOW_END = pd.to_datetime('08:30:00').time()
MORNING_START = pd.to_datetime('04:00:00').time()
MORNING_END = pd.to_datetime('07:30:00').time()
MORNING_SPLIT = (pd.Timestamp('1900-01-01 04:00:00') +
                 (pd.Timestamp('1900-01-01 07:30:00') - pd.Timestamp('1900-01-01 04:00:00')) / 2).time()
MIN_MORNING_WINDOW_COVERAGE = 0.50
MIN_AM_WINDOW_COVERAGE = 0.50

FORECAST_PARAM_GRID = {
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.03, 0.05, 0.08, 0.12],
    'subsample': [0.70, 0.85, 1.00],
    'colsample_bytree': [0.60, 0.80, 1.00],
    'min_child_weight': [1, 3, 5],
    'reg_lambda': [1, 3, 5],
}

DRIVE_EXPORT_DIR = '/content/drive/MyDrive/AH2179 GP5/prediction model v8.2 (portals 7-8, time-aware)'
FINAL_ARTIFACT_FILENAME = 'fS_prediction_model_p78_FINAL_cmax_v82.joblib'

# First matching artifact is used. Keep the specific v8.2 name first.
CMAX_MODEL_CANDIDATE_PATHS = [
    os.path.join(DRIVE_EXPORT_DIR, 'cmax_prediction_model_p78_v82.joblib'),
]

EXPECTED_CMAX_FEATURES = [
    'v', 'f', 'c_early', 'f_trend_ratio_log', 'v_trend_delta',
    'day_of_week', 'is_weekend',
]

# =============================================================================
# MODEL LOADING AND VALIDATION
# =============================================================================

def find_first_existing(paths):
    for path in paths:
        if os.path.isfile(path):
            return path
    return None


def load_cmax_v82_artifact():
    path = find_first_existing(CMAX_MODEL_CANDIDATE_PATHS)
    if path is None:
        raise FileNotFoundError(
            'Could not find the C_max v8.2 artifact. Checked:\n  ' +
            '\n  '.join(CMAX_MODEL_CANDIDATE_PATHS) +
            '\n\nSave/upload the new C_max artifact, or add its exact path to '
            'CMAX_MODEL_CANDIDATE_PATHS.'
        )

    artifact = joblib.load(path)
    required_keys = {'portal_models', 'feature_order'}
    missing = required_keys - set(artifact.keys())
    if missing:
        raise KeyError(f'C_max artifact at {path} is missing required key(s): {sorted(missing)}')

    feature_order = list(artifact['feature_order'])
    if feature_order != EXPECTED_CMAX_FEATURES:
        raise ValueError(
            'This script is specifically compatible with the supplied C_max v8.2 '
            'feature contract.\n'
            f'Expected: {EXPECTED_CMAX_FEATURES}\n'
            f'Artifact: {feature_order}\n'
            'Do not continue: S_pred would be calculated inconsistently.'
        )

    portal_models = artifact['portal_models']
    missing_portals = [p for p in ACTIVE_PORTALS if p not in portal_models]
    if missing_portals:
        raise KeyError(f'C_max artifact lacks portal model(s): {missing_portals}')

    print(f'[Load] C_max v8.2 artifact: {path}')
    print(f'[Load] Validated capacity feature contract: {feature_order}')
    return artifact


capacity_artifacts = load_cmax_v82_artifact()
capacity_models = capacity_artifacts['portal_models']
CAPACITY_FEATURE_COLUMNS = list(capacity_artifacts['feature_order'])

# =============================================================================
# DAILY C_max PREDICTION -- MATCHES C_max v8.2 FEATURE CONTRACT
# =============================================================================

def expected_rows(start_time, end_time, interval_minutes):
    duration_minutes = (
        pd.Timestamp(f'1900-01-01 {end_time}') -
        pd.Timestamp(f'1900-01-01 {start_time}')
    ).total_seconds() / 60.0
    return int(duration_minutes // interval_minutes) + 1


def add_capacity_calendar_features(features_df):
    """Adds the exact two C_max v8.2 calendar fields.

    Date is a Python date after groupby/reset_index. Convert to pandas
    datetime before using weekday properties. Values intentionally match
    the C_max v8.2 convention: Monday=0 through Sunday=6; weekend=1.
    """
    date_series = pd.to_datetime(features_df['Date'])
    features_df['day_of_week'] = date_series.dt.dayofweek.astype(int)
    features_df['is_weekend'] = (date_series.dt.dayofweek >= 5).astype(int)
    return features_df


def estimate_daily_capacity_v82(seq, raw_df, verbose=True):
    """Calculates daily C_max_pred using the new seven-feature C_max v8.2
    model. This is the critical compatibility layer between Part 1 v8.2
    and Part 2: calendar features are reconstructed before prediction."""
    p_df = raw_df[raw_df[RAW_COLUMNS['sequence']] == seq].copy()
    if p_df.empty:
        raise ValueError(f'Portal {seq} has no rows in the supplied data.')

    p_df['Date'] = p_df[RAW_COLUMNS['datetime']].dt.date
    p_df['Time'] = p_df[RAW_COLUMNS['datetime']].dt.time
    p_df['C'] = p_df[RAW_COLUMNS['speed']] * p_df[RAW_COLUMNS['flow']]

    mask_morning = (p_df['Time'] >= MORNING_START) & (p_df['Time'] < MORNING_END)
    morning_counts = p_df.loc[mask_morning].groupby('Date').size()
    expected = expected_rows(MORNING_START, MORNING_END, INTERVAL_MINUTES)

    all_days = set(p_df['Date'].unique())
    valid_days = set(morning_counts[morning_counts >= MIN_MORNING_WINDOW_COVERAGE * expected].index)
    if verbose and len(all_days - valid_days):
        print(f'  [C_max v8.2, portal {seq}] Dropped {len(all_days - valid_days)} day(s) below morning coverage threshold.')

    p_df = p_df[p_df['Date'].isin(valid_days)].copy()
    early = p_df[(p_df['Time'] >= MORNING_START) & (p_df['Time'] < MORNING_END)]

    daily_v = early.groupby('Date')[RAW_COLUMNS['speed']].mean().rename('v')
    daily_f = early.groupby('Date')[RAW_COLUMNS['flow']].sum().rename('f')
    daily_c_early = early.groupby('Date')['C'].quantile(0.98).rename('c_early')

    first_half = p_df[(p_df['Time'] >= MORNING_START) & (p_df['Time'] < MORNING_SPLIT)]
    second_half = p_df[(p_df['Time'] >= MORNING_SPLIT) & (p_df['Time'] < MORNING_END)]

    f_first = first_half.groupby('Date')[RAW_COLUMNS['flow']].sum().rename('f_first_half')
    f_second = second_half.groupby('Date')[RAW_COLUMNS['flow']].sum().rename('f_second_half')
    flow_halves = pd.concat([f_first, f_second], axis=1).fillna(0.0)
    ratio = (flow_halves['f_second_half'] + 1.0) / (flow_halves['f_first_half'] + 1.0)
    f_trend_ratio_log = np.log(ratio).rename('f_trend_ratio_log')

    v_first = first_half.groupby('Date')[RAW_COLUMNS['speed']].mean().rename('v_first_half')
    v_second = second_half.groupby('Date')[RAW_COLUMNS['speed']].mean().rename('v_second_half')
    v_trend_delta = (v_second.fillna(0.0) - v_first.fillna(0.0)).rename('v_trend_delta')

    features_df = pd.concat(
        [daily_v, daily_f, daily_c_early, f_trend_ratio_log, v_trend_delta], axis=1
    ).reset_index().rename(columns={'index': 'Date'}).dropna()
    features_df = add_capacity_calendar_features(features_df)

    missing = [c for c in CAPACITY_FEATURE_COLUMNS if c not in features_df.columns]
    if missing:
        raise KeyError(f'Capacity-feature construction failed; missing {missing}')

    entry = capacity_models[seq]
    x_capacity = features_df[CAPACITY_FEATURE_COLUMNS].to_numpy(dtype=float)
    x_capacity_scaled = entry['scaler'].transform(x_capacity)
    features_df['C_max_pred'] = entry['model'].predict(x_capacity_scaled)

    if verbose:
        print(f'  [C_max v8.2, portal {seq}] Produced C_max_pred for {len(features_df)} of {len(all_days)} day(s).')
    return features_df[['Date', 'C_max_pred']]

# =============================================================================
# FEATURE ENGINEERING -- SAME 18 PART 2 INPUTS AS BEFORE
# =============================================================================

def add_temporal_features(df):
    generated = []
    for var in BASE_VARS:
        for lag in LAG_STEP_LIST:
            name = f'{var}_lag{lag}'
            df[name] = df[var].shift(lag).bfill()
            generated.append(name)
    return df, generated


def build_corridor_dataframe(raw_df):
    daily_capacity = {seq: estimate_daily_capacity_v82(seq, raw_df) for seq in ACTIVE_PORTALS}
    portal_frames = {}

    for seq in ACTIVE_PORTALS:
        portal_raw = raw_df[raw_df[RAW_COLUMNS['sequence']] == seq].copy()
        portal_raw = portal_raw.rename(columns={
            RAW_COLUMNS['datetime']: 'DateTime',
            RAW_COLUMNS['speed']: 'v',
            RAW_COLUMNS['flow']: 'f',
        }).set_index('DateTime')

        resampled = portal_raw.resample(f'{INTERVAL_MINUTES}min').agg({
            'v': lambda x: x[x > 0].mean() if (x > 0).any() else 0.0,
            'f': 'sum',
        }).reset_index()
        resampled['Date'] = resampled['DateTime'].dt.date
        resampled['C'] = resampled['v'] * resampled['f']
        resampled = resampled.merge(daily_capacity[seq], on='Date', how='inner')
        resampled['S_pred'] = resampled['C'] / resampled['C_max_pred']
        resampled = resampled.sort_values('DateTime').reset_index(drop=True)
        resampled, lag_cols = add_temporal_features(resampled)

        rename = {'v': f'p{seq}_v', 'f': f'p{seq}_f', 'S_pred': f'p{seq}_S_pred'}
        rename.update({column: f'p{seq}_{column}' for column in lag_cols})
        resampled = resampled.rename(columns=rename)

        keep = ['DateTime', 'Date', f'p{seq}_v', f'p{seq}_f', f'p{seq}_S_pred'] + [f'p{seq}_{c}' for c in lag_cols]
        portal_frames[seq] = resampled[keep]

    corridor = portal_frames[ACTIVE_PORTALS[0]]
    for seq in ACTIVE_PORTALS[1:]:
        corridor = corridor.merge(portal_frames[seq], on=['DateTime', 'Date'], how='inner')
    return corridor.sort_values('DateTime').reset_index(drop=True)


def align_valid_days(corridor_df):
    expected = expected_rows(AM_WINDOW_START, AM_WINDOW_END, INTERVAL_MINUTES)
    time_values = corridor_df['DateTime'].dt.time
    am_mask = (time_values >= AM_WINDOW_START) & (time_values <= AM_WINDOW_END)
    counts = corridor_df.loc[am_mask].groupby('Date').size()
    valid_days = set(counts[counts >= MIN_AM_WINDOW_COVERAGE * expected].index)
    before = corridor_df['Date'].nunique()
    corridor_df = corridor_df[corridor_df['Date'].isin(valid_days)].reset_index(drop=True)
    print(f'[Day alignment] {before - corridor_df["Date"].nunique()} day(s) removed; {corridor_df["Date"].nunique()} valid day(s) remain.')
    return corridor_df


def build_feature_columns(target_name):
    columns = []
    for seq in ACTIVE_PORTALS:
        columns += [f'p{seq}_{var}' for var in BASE_VARS]
        columns += [f'p{seq}_{var}_lag{lag}' for var in BASE_VARS for lag in LAG_STEP_LIST]
    if len(columns) != 18:
        raise RuntimeError(f'Expected 18 Part 2 inputs; built {len(columns)}: {columns}')
    return columns

# =============================================================================
# TARGET PREPARATION
# =============================================================================

def add_smoothed_target(corridor_df, target_name, window):
    raw_column = f'p{TARGET_PORTAL_SEQ}_{target_name}'
    smooth_column = f'{raw_column}_smoothed_w{window}'
    if window <= 1:
        corridor_df[smooth_column] = corridor_df[raw_column]
    else:
        corridor_df[smooth_column] = corridor_df.groupby('Date')[raw_column].transform(
            lambda values: values.rolling(window=window, center=True, min_periods=1).mean()
        )
    return smooth_column


def build_samples(corridor_df, feature_columns, target_name, config):
    target_level_column = add_smoothed_target(corridor_df, target_name, config['smooth_window'])
    current_raw_column = f'p{TARGET_PORTAL_SEQ}_{target_name}'

    x_rows, y_rows, current_values, dates = [], [], [], []
    for row_idx in range(MAX_LAG_NEEDED, len(corridor_df) - FORECAST_STEPS):
        timestamp = corridor_df.loc[row_idx, 'DateTime']
        if not (AM_WINDOW_START <= timestamp.time() <= AM_WINDOW_END):
            continue
        if corridor_df.loc[row_idx, 'Date'] != corridor_df.loc[row_idx + FORECAST_STEPS, 'Date']:
            continue

        x_rows.append(corridor_df.loc[row_idx, feature_columns].to_numpy(dtype=float))
        current = float(corridor_df.loc[row_idx, current_raw_column])
        future_levels = corridor_df.loc[row_idx + 1:row_idx + FORECAST_STEPS, target_level_column].to_numpy(dtype=float)
        y_rows.append(future_levels - current if config['predict_delta'] else future_levels)
        current_values.append(current)
        dates.append(corridor_df.loc[row_idx, 'Date'])

    samples = {
        'X': np.asarray(x_rows),
        'y': np.asarray(y_rows),
        'current': np.asarray(current_values),
        'dates': np.asarray(dates),
    }
    if samples['X'].shape[0] == 0:
        raise ValueError(f'No usable samples created for target {target_name}.')
    return samples

# =============================================================================
# SEARCH, CROSS-VALIDATION, AND FINAL FIT
# =============================================================================

def fit_early_stopped_xgb(params, x_train, y_train, x_val, y_val):
    model = xgb.XGBRegressor(
        **params,
        n_estimators=MAX_N_ESTIMATORS_WITH_EARLY_STOP,
        random_state=RANDOM_STATE,
        eval_metric='rmse',
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )
    model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    return model


def search_score(x, y, dates, params):
    unique_days = np.asarray(sorted(pd.unique(dates)))
    folds = min(SEARCH_CV_FOLDS, max(2, len(unique_days) - 1))
    edges = np.linspace(0, len(unique_days), folds + 1, dtype=int)
    scores = []

    for fold in range(folds - 1):
        train_days = set(unique_days[:edges[fold + 1]])
        test_days = set(unique_days[edges[fold + 1]:edges[fold + 2]])
        fit_days_sorted = sorted(train_days)
        if not test_days or len(fit_days_sorted) < 2:
            continue

        n_validation_days = max(1, int(0.15 * len(fit_days_sorted)))
        validation_days = set(fit_days_sorted[-n_validation_days:])
        fitting_days = set(fit_days_sorted[:-n_validation_days])
        if not fitting_days:
            continue

        fit_mask = np.asarray([day in fitting_days for day in dates])
        val_mask = np.asarray([day in validation_days for day in dates])
        test_mask = np.asarray([day in test_days for day in dates])
        if fit_mask.sum() < 10 or val_mask.sum() < 1 or test_mask.sum() < 1:
            continue

        scaler = StandardScaler()
        x_fit = scaler.fit_transform(x[fit_mask])
        x_val = scaler.transform(x[val_mask])
        x_test = scaler.transform(x[test_mask])
        model = fit_early_stopped_xgb(params, x_fit, y[fit_mask], x_val, y[val_mask])
        scores.append(r2_score(y[test_mask], model.predict(x_test)))

    return float(np.mean(scores)) if scores else np.nan


def search_best_params(x, y_all_horizons, dates, target_name):
    candidates = list(ParameterSampler(FORECAST_PARAM_GRID, n_iter=N_SEARCH_ITER, random_state=RANDOM_STATE))
    best_by_horizon = {}
    search_rows = []

    print(f'\n[Search] target={target_name}; {FORECAST_STEPS} horizons; {N_SEARCH_ITER} candidates each.')
    for step in range(FORECAST_STEPS):
        best_score, best_params = -np.inf, None
        for candidate_idx, params in enumerate(candidates):
            score = search_score(x, y_all_horizons[:, step], dates, params)
            search_rows.append({'Target': target_name, 'Horizon_min': step + 1, 'Candidate': candidate_idx, 'CV_R2': score, **params})
            if not np.isnan(score) and score > best_score:
                best_score, best_params = score, params
        best_by_horizon[step] = best_params or {'max_depth': 4, 'learning_rate': 0.05}
        print(f'  +{step + 1:02d} min | inner-CV R2={best_score:.4f}')
    return best_by_horizon, pd.DataFrame(search_rows)


def evaluate_repeated_cv(x, y, current, dates, params_by_horizon, target_name, predict_delta):
    unique_days = np.asarray(sorted(pd.unique(dates)))
    rows = []
    for step in range(FORECAST_STEPS):
        for repeat in range(N_CV_REPEATS):
            shuffled = unique_days.copy()
            np.random.RandomState(RANDOM_STATE + repeat).shuffle(shuffled)
            edges = np.linspace(0, len(shuffled), N_CV_FOLDS + 1, dtype=int)

            for fold in range(N_CV_FOLDS):
                test_days = set(shuffled[edges[fold]:edges[fold + 1]])
                train_days = set(shuffled) - test_days
                train_mask = np.asarray([day in train_days for day in dates])
                test_mask = np.asarray([day in test_days for day in dates])
                if train_mask.sum() == 0 or test_mask.sum() == 0:
                    continue

                scaler = StandardScaler()
                x_train = scaler.fit_transform(x[train_mask])
                x_test = scaler.transform(x[test_mask])
                model = xgb.XGBRegressor(**params_by_horizon[step], random_state=RANDOM_STATE)
                model.fit(x_train, y[train_mask, step])
                predicted = model.predict(x_test)
                actual = y[test_mask, step]

                if predict_delta:
                    predicted = predicted + current[test_mask]
                    actual = actual + current[test_mask]

                rows.append({
                    'Metric Target': target_name,
                    'Horizon (min)': step + 1,
                    'Repeat': repeat + 1,
                    'Fold': fold + 1,
                    'R2 Score': r2_score(actual, predicted),
                    'MAE': mean_absolute_error(actual, predicted),
                    'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
                })
    return pd.DataFrame(rows)


def fit_final_models(x, y, params_by_horizon, feature_columns, target_config):
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(x)
    models = {}
    for step in range(FORECAST_STEPS):
        model = xgb.XGBRegressor(**params_by_horizon[step], random_state=RANDOM_STATE)
        model.fit(x_scaled, y[:, step])
        models[step] = model
    return {
        'models': models,
        'scaler': scaler,
        'feature_cols': feature_columns,
        **target_config,
    }

# =============================================================================
# DEPLOYMENT INFERENCE HELPER
# =============================================================================

def predict_level(horizon_models, target_name, x_row, current_value):
    """Returns +1 through +15 minute LEVEL predictions.

    This helper is mandatory for S_pred because that final target is trained
    as a delta. It automatically adds current_value only when the artifact's
    per-target configuration says predict_delta=True.
    """
    entry = horizon_models[target_name]
    x_scaled = entry['scaler'].transform(np.asarray(x_row, dtype=float))
    predictions = np.asarray([
        entry['models'][step].predict(x_scaled)[0]
        for step in range(FORECAST_STEPS)
    ])
    if entry['predict_delta']:
        predictions = predictions + current_value
    return predictions

# =============================================================================
# MAIN
# =============================================================================

if __name__ == '__main__':
    if 'final_df' not in globals():
        raise NameError('final_df is not available. Run preprocessing first; use final_df, not portal_min_df.')

    raw_df = final_df.copy()
    if not pd.api.types.is_datetime64_any_dtype(raw_df[RAW_COLUMNS['datetime']]):
        raw_df[RAW_COLUMNS['datetime']] = pd.to_datetime(raw_df[RAW_COLUMNS['datetime']])

    detected_portals = set(raw_df[RAW_COLUMNS['sequence']].astype(int).unique())
    missing_data_portals = [portal for portal in ACTIVE_PORTALS if portal not in detected_portals]
    if missing_data_portals:
        raise ValueError(f'Input data lacks required portal(s): {missing_data_portals}; found {sorted(detected_portals)}')

    print('\n' + '=' * 80)
    print('PART 2 FINAL -- RETRAINING AGAINST C_max v8.2')
    print('=' * 80)
    print(f'Portal inputs: {ACTIVE_PORTALS}; target portal: {TARGET_PORTAL_SEQ}')
    print(f'Part 2 variables/lags: {BASE_VARS}; current + lags {LAG_STEP_LIST}; 18 inputs total')
    print(f'Final target configuration: {TARGET_TRAINING_CONFIG}')

    print('\n' + '=' * 80 + '\nSTEP 1: FEATURE ENGINEERING WITH C_max v8.2\n' + '=' * 80)
    corridor_df = build_corridor_dataframe(raw_df)
    corridor_df = align_valid_days(corridor_df)
    print(f'[Feature engineering] corridor_df shape: {corridor_df.shape}')

    feature_cols_by_target = {target: build_feature_columns(target) for target in ACTIVE_FORECAST_TARGETS}
    for target, columns in feature_cols_by_target.items():
        print(f'[Inputs] target={target}: {len(columns)} columns: {columns}')

    print('\n' + '=' * 80 + '\nSTEP 2: SEARCH, REPEATED CV, AND FINAL FIT\n' + '=' * 80)
    horizon_models = {}
    cv_frames = []
    search_frames = []

    for target in ACTIVE_FORECAST_TARGETS:
        config = TARGET_TRAINING_CONFIG[target]
        feature_columns = feature_cols_by_target[target]
        samples = build_samples(corridor_df, feature_columns, target, config)
        print(f'\n[Target {target}] samples: X={samples["X"].shape}; config={config}')

        params_by_horizon, search_df = search_best_params(samples['X'], samples['y'], samples['dates'], target)
        search_frames.append(search_df)

        cv_df = evaluate_repeated_cv(
            samples['X'], samples['y'], samples['current'], samples['dates'],
            params_by_horizon, target, config['predict_delta']
        )
        cv_frames.append(cv_df)
        horizon_models[target] = fit_final_models(
            samples['X'], samples['y'], params_by_horizon, feature_columns, config
        )
        print(f'[Final fit] target={target}: {FORECAST_STEPS} horizon models trained on {len(samples["X"])} samples.')

    cv_detail_df = pd.concat(cv_frames, ignore_index=True)
    cv_summary = cv_detail_df.groupby(['Metric Target', 'Horizon (min)'])[['R2 Score', 'MAE', 'RMSE']].agg(['mean', 'std']).round(4)
    cv_detail_df.to_csv('output/cv_fold_detail_log_p78_FINAL_cmax_v82.csv', index=False)
    pd.concat(search_frames, ignore_index=True).to_csv('output/hyperparam_search_log_p78_FINAL_cmax_v82.csv', index=False)

    print('\n' + '=' * 80)
    print('FINAL REPEATED 5-FOLD CV SUMMARY -- C_max v8.2 COMPATIBLE PART 2')
    print('=' * 80)
    print(cv_summary.to_string())

    forecast_artifact = {
        'trained_models': horizon_models,
        'forecast_steps': FORECAST_STEPS,
        'interval_minutes': INTERVAL_MINUTES,
        'target_portal_seq': TARGET_PORTAL_SEQ,
        'active_portals': ACTIVE_PORTALS,
        'lag_step_list': LAG_STEP_LIST,
        'active_forecast_targets': ACTIVE_FORECAST_TARGETS,
        'target_training_config': TARGET_TRAINING_CONFIG,
        'capacity_model_feature_order': CAPACITY_FEATURE_COLUMNS,
        'capacity_model_version': 'v8.2 calendar-aware C_max',
        'description': (
            'Final Part 2 model retrained against C_max v8.2. C_max feature order is '
            f'{CAPACITY_FEATURE_COLUMNS}. Part 2 inputs remain only portal-7/8 '
            '[v,f,S_pred] current values plus lag1/lag2. f uses a 5-minute centered '
            'smoothed level target; S_pred uses a 5-minute centered smoothed delta target. '
            'Use predict_level() to recover level predictions, particularly for S_pred.'
        ),
    }

    drive_path = os.path.join(DRIVE_EXPORT_DIR, FINAL_ARTIFACT_FILENAME)
    try:
        os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)
        joblib.dump(forecast_artifact, drive_path)
        exported_path = drive_path
    except (FileNotFoundError, OSError) as error:
        exported_path = os.path.join('output', FINAL_ARTIFACT_FILENAME)
        joblib.dump(forecast_artifact, exported_path)
        print(f'\n[Export warning] Drive unavailable: {error}')
        print('[Export warning] Saved local fallback; download it and upload manually.')

    print('\n' + '=' * 80)
    print('COMPLETE')
    print('=' * 80)
    print(f'[Export] Final artifact: {exported_path}')
    print("[Deployment] f: use all +1..+15 minute horizons.")
    print("[Deployment] S_pred: deploy +1..+8 minutes; later horizons remain informational only.")
    print('[Deployment] Always call predict_level() rather than calling a raw XGBoost model directly.')

[Load] C_max v8.2 artifact: /content/drive/MyDrive/AH2179 GP5/prediction model v8.2 (portals 7-8, time-aware)/cmax_prediction_model_p78_v82.joblib
[Load] Validated capacity feature contract: ['v', 'f', 'c_early', 'f_trend_ratio_log', 'v_trend_delta', 'day_of_week', 'is_weekend']

PART 2 FINAL -- RETRAINING AGAINST C_max v8.2
Portal inputs: [7, 8]; target portal: 8
Part 2 variables/lags: ['v', 'f', 'S_pred']; current + lags [1, 2]; 18 inputs total
Final target configuration: {'f': {'smooth_window': 5, 'predict_delta': False, 'documented_reliable_through_min': 15}, 'S_pred': {'smooth_window': 5, 'predict_delta': True, 'documented_reliable_through_min': 8}}

STEP 1: FEATURE ENGINEERING WITH C_max v8.2
  [C_max v8.2, portal 7] Produced C_max_pred for 214 of 214 day(s).
  [C_max v8.2, portal 8] Produced C_max_pred for 214 of 214 day(s).
[Day alignment] 0 day(s) removed; 214 valid day(s) remain.
[Feature engineering] corridor_df shape: (307081, 20)
[Inputs] target=f: 18 columns: ['p7_v', 'p7

## Evaluation with train/test split

### Metrics

In [ ]:
"""
Final Part 2 Evaluation + Diagnostics -- C_max v8.2-Compatible Forecaster
================================================================================
Loads the FINAL Part 2 model trained against C_max v8.2, rebuilds the exact
capacity-aware [v, f, S_pred] + lag1/lag2 input matrix on a separate
EVALUATION/HOLDOUT dataset, produces predictions for all 15 horizons, and
exports detailed metrics and diagnostics.

REQUIRED ARTIFACTS
------------------
1. C_max v8.2 artifact with feature order:
   [v, f, c_early, f_trend_ratio_log, v_trend_delta, day_of_week, is_weekend]
2. Final Part 2 artifact:
   fS_prediction_model_p78_FINAL_cmax_v82.joblib

EVALUATION DATA
---------------
Recommended: load separately preprocessed holdout data into `final_df_eval`
before running this script. Required columns:
    sequence, DateTime, v, f

Alternatively set EVAL_CSV_PATH below to a CSV path. `final_df` is accepted
only as a last resort and emits a warning because it may contain training data.

IMPORTANT: Results are a true external evaluation only if eval data contains
dates that the final Part 2 training and hyperparameter search never used.

OUTPUTS
-------
output/evaluation_final_cmax_v82/
    evaluation_horizon_metrics.csv
    evaluation_prediction_detail.csv
    evaluation_target_summary.csv
    evaluation_segment_metrics.csv
    evaluation_day_metrics.csv
    evaluation_high_error_origins.csv
    evaluation_calibration.csv
    r2_by_horizon.png
    mae_rmse_by_horizon.png
    predicted_vs_actual_scatter.png
    residual_by_horizon.png
"""

import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# =============================================================================
# CONFIGURATION
# =============================================================================

RAW_COLUMNS = {'datetime': 'DateTime', 'sequence': 'sequence', 'speed': 'v', 'flow': 'f'}
DRIVE_EXPORT_DIR = '/content/drive/MyDrive/AH2179 GP5/prediction model v8.2 (portals 7-8, time-aware)'

FORECASTER_CANDIDATES = [
    os.path.join(DRIVE_EXPORT_DIR, 'fS_prediction_model_p78_FINAL_cmax_v82.joblib'),
    os.path.join('output', 'fS_prediction_model_p78_FINAL_cmax_v82.joblib'),
    'fS_prediction_model_p78_FINAL_cmax_v82.joblib',
]
CMAX_CANDIDATES = [
    os.path.join(DRIVE_EXPORT_DIR, 'cmax_prediction_model_p78_v82.joblib'),
    os.path.join(DRIVE_EXPORT_DIR, 'cmax_prediction_model_v82_p78.joblib'),
    os.path.join(DRIVE_EXPORT_DIR, 'cmax_prediction_model_p78.joblib'),
    os.path.join('output', 'cmax_prediction_model_p78_v82.joblib'),
    os.path.join('output', 'cmax_prediction_model_v82_p78.joblib'),
    os.path.join('output', 'cmax_prediction_model_p78.joblib'),
]

# Set this only when evaluation data is stored as a separate CSV.
# Example: EVAL_CSV_PATH = '/content/evaluation_dataset.csv'
EVAL_CSV_PATH = None

# Optional safety filter if you only have final_df. These are inclusive bounds.
# Example: EVAL_DATE_START = '2022-01-01'; EVAL_DATE_END = '2022-03-31'
# Leave both None when using a separate final_df_eval / evaluation CSV.
EVAL_DATE_START = None
EVAL_DATE_END = None

OUTPUT_DIR = 'output/evaluation_final_cmax_v82'
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXPECTED_CMAX_FEATURES = [
    'v', 'f', 'c_early', 'f_trend_ratio_log', 'v_trend_delta',
    'day_of_week', 'is_weekend',
]

# =============================================================================
# LOAD ARTIFACTS
# =============================================================================

def find_file(paths):
    for path in paths:
        if os.path.isfile(path):
            return path
    return None


def load_artifacts():
    forecaster_path = find_file(FORECASTER_CANDIDATES)
    cmax_path = find_file(CMAX_CANDIDATES)
    if forecaster_path is None:
        raise FileNotFoundError('Final forecaster not found. Checked:\n  ' + '\n  '.join(FORECASTER_CANDIDATES))
    if cmax_path is None:
        raise FileNotFoundError('C_max v8.2 artifact not found. Checked:\n  ' + '\n  '.join(CMAX_CANDIDATES))

    forecaster = joblib.load(forecaster_path)
    cmax = joblib.load(cmax_path)

    required_forecaster = {'trained_models', 'forecast_steps', 'interval_minutes', 'target_portal_seq', 'active_portals'}
    missing_forecaster = required_forecaster - set(forecaster)
    if missing_forecaster:
        raise KeyError(f'Forecaster artifact missing: {sorted(missing_forecaster)}')

    if list(cmax.get('feature_order', [])) != EXPECTED_CMAX_FEATURES:
        raise ValueError(
            'Loaded C_max artifact does not match the required v8.2 calendar-aware feature contract.\n'
            f'Expected: {EXPECTED_CMAX_FEATURES}\nFound: {cmax.get("feature_order")}'
        )

    print(f'[Load] Forecaster: {forecaster_path}')
    print(f'[Load] C_max v8.2: {cmax_path}')
    return forecaster, cmax


forecaster_artifact, cmax_artifact = load_artifacts()
trained_models = forecaster_artifact['trained_models']
capacity_models = cmax_artifact['portal_models']
CAPACITY_FEATURE_COLUMNS = cmax_artifact['feature_order']

FORECAST_STEPS = int(forecaster_artifact['forecast_steps'])
INTERVAL_MINUTES = int(forecaster_artifact['interval_minutes'])
TARGET_PORTAL_SEQ = int(forecaster_artifact['target_portal_seq'])
ACTIVE_PORTALS = list(forecaster_artifact['active_portals'])
ACTIVE_TARGETS = list(forecaster_artifact.get('active_forecast_targets', trained_models.keys()))
ACTIVE_TARGETS = [target for target in ACTIVE_TARGETS if target in trained_models]
LAG_STEPS = list(forecaster_artifact.get('lag_step_list', [1, 2]))
MAX_LAG = max(LAG_STEPS)

AM_WINDOW_START = pd.Timestamp('1900-01-01 07:30').time()
AM_WINDOW_END = pd.Timestamp('1900-01-01 08:30').time()
MORNING_START = pd.Timestamp('1900-01-01 04:00').time()
MORNING_END = pd.Timestamp('1900-01-01 07:30').time()
MORNING_SPLIT = pd.Timestamp('1900-01-01 05:45').time()
MIN_COVERAGE = 0.50

print(f'[Load] Portals={ACTIVE_PORTALS}; target portal={TARGET_PORTAL_SEQ}; targets={ACTIVE_TARGETS}')
for target in ACTIVE_TARGETS:
    entry = trained_models[target]
    print(f"  {target}: smooth_window={entry.get('smooth_window', 1)}, predict_delta={entry.get('predict_delta', False)}, features={len(entry['feature_cols'])}")

# =============================================================================
# LOAD EVALUATION DATA
# =============================================================================

def load_evaluation_data():
    if 'final_df_eval' in globals():
        print('[Data] Using in-memory final_df_eval (recommended external holdout).')
        df = globals()['final_df_eval'].copy()
    elif EVAL_CSV_PATH is not None:
        if not os.path.isfile(EVAL_CSV_PATH):
            raise FileNotFoundError(f'EVAL_CSV_PATH does not exist: {EVAL_CSV_PATH}')
        print(f'[Data] Loading evaluation CSV: {EVAL_CSV_PATH}')
        df = pd.read_csv(EVAL_CSV_PATH)
    elif 'final_df' in globals():
        print('[Data warning] Using final_df. Confirm these dates were not used for final training.')
        df = globals()['final_df'].copy()
    else:
        raise RuntimeError(
            'No evaluation data found. Load holdout data as final_df_eval, or set EVAL_CSV_PATH. '
            'final_df is unavailable in this session.'
        )

    missing = [column for column in ['sequence', 'DateTime', 'v', 'f'] if column not in df.columns]
    if missing:
        raise KeyError(f'Evaluation data missing required columns: {missing}')

    df['DateTime'] = pd.to_datetime(df['DateTime'])
    if EVAL_DATE_START is not None:
        df = df[df['DateTime'] >= pd.Timestamp(EVAL_DATE_START)]
    if EVAL_DATE_END is not None:
        df = df[df['DateTime'] <= pd.Timestamp(EVAL_DATE_END) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)]

    available = set(df['sequence'].astype(int).unique())
    missing_portals = [portal for portal in ACTIVE_PORTALS if portal not in available]
    if missing_portals:
        raise ValueError(f'Evaluation data lacks portal(s) {missing_portals}; found {sorted(available)}')
    if df.empty:
        raise ValueError('No rows remain after evaluation-date filtering.')

    print(f'[Data] Rows={len(df):,}; dates={df.DateTime.min()} to {df.DateTime.max()}; days={df.DateTime.dt.date.nunique()}')
    return df


eval_raw_df = load_evaluation_data()

# =============================================================================
# C_max v8.2 AND PART 2 FEATURE RECONSTRUCTION
# =============================================================================

def expected_rows(start, end):
    minutes = (pd.Timestamp(f'1900-01-01 {end}') - pd.Timestamp(f'1900-01-01 {start}')).total_seconds() / 60
    return int(minutes // INTERVAL_MINUTES) + 1


def estimate_daily_capacity(seq, raw_df):
    portal = raw_df[raw_df['sequence'] == seq].copy()
    portal['Date'] = portal['DateTime'].dt.date
    portal['Time'] = portal['DateTime'].dt.time
    portal['C'] = portal['v'] * portal['f']

    morning_mask = (portal.Time >= MORNING_START) & (portal.Time < MORNING_END)
    morning_counts = portal.loc[morning_mask].groupby('Date').size()
    expected = expected_rows(MORNING_START, MORNING_END)
    valid_days = set(morning_counts[morning_counts >= MIN_COVERAGE * expected].index)
    all_days = set(portal.Date.unique())
    portal = portal[portal.Date.isin(valid_days)].copy()

    early = portal[(portal.Time >= MORNING_START) & (portal.Time < MORNING_END)]
    first = portal[(portal.Time >= MORNING_START) & (portal.Time < MORNING_SPLIT)]
    second = portal[(portal.Time >= MORNING_SPLIT) & (portal.Time < MORNING_END)]

    v = early.groupby('Date')['v'].mean().rename('v')
    f = early.groupby('Date')['f'].sum().rename('f')
    c_early = early.groupby('Date')['C'].quantile(0.98).rename('c_early')
    f_first = first.groupby('Date')['f'].sum().rename('first')
    f_second = second.groupby('Date')['f'].sum().rename('second')
    f_halves = pd.concat([f_first, f_second], axis=1).fillna(0.0)
    f_ratio = np.log((f_halves['second'] + 1.0) / (f_halves['first'] + 1.0)).rename('f_trend_ratio_log')
    v_first = first.groupby('Date')['v'].mean().rename('first')
    v_second = second.groupby('Date')['v'].mean().rename('second')
    v_delta = (v_second.fillna(0.0) - v_first.fillna(0.0)).rename('v_trend_delta')

    features = pd.concat([v, f, c_early, f_ratio, v_delta], axis=1).reset_index().dropna()
    calendar_dates = pd.to_datetime(features['Date'])
    features['day_of_week'] = calendar_dates.dt.dayofweek.astype(int)
    features['is_weekend'] = (calendar_dates.dt.dayofweek >= 5).astype(int)

    entry = capacity_models[seq]
    x = features[CAPACITY_FEATURE_COLUMNS].to_numpy(dtype=float)
    features['C_max_pred'] = entry['model'].predict(entry['scaler'].transform(x))
    print(f'  [C_max] portal {seq}: {len(features)} of {len(all_days)} day(s) yielded C_max_pred.')
    return features[['Date', 'C_max_pred']]


def add_lags(frame):
    for variable in ['v', 'f', 'S_pred']:
        for lag in LAG_STEPS:
            frame[f'{variable}_lag{lag}'] = frame[variable].shift(lag).bfill()
    return frame


def build_eval_corridor(raw_df):
    capacities = {seq: estimate_daily_capacity(seq, raw_df) for seq in ACTIVE_PORTALS}
    frames = {}
    for seq in ACTIVE_PORTALS:
        portal = raw_df[raw_df.sequence == seq].copy().set_index('DateTime')
        resampled = portal.resample(f'{INTERVAL_MINUTES}min').agg({
            'v': lambda x: x[x > 0].mean() if (x > 0).any() else 0.0,
            'f': 'sum',
        }).reset_index()
        resampled['Date'] = resampled.DateTime.dt.date
        resampled['C'] = resampled['v'] * resampled['f']
        resampled = resampled.merge(capacities[seq], on='Date', how='inner')
        resampled['S_pred'] = resampled['C'] / resampled['C_max_pred']
        resampled = add_lags(resampled.sort_values('DateTime').reset_index(drop=True))

        rename = {variable: f'p{seq}_{variable}' for variable in ['v', 'f', 'S_pred']}
        rename.update({f'{variable}_lag{lag}': f'p{seq}_{variable}_lag{lag}' for variable in ['v', 'f', 'S_pred'] for lag in LAG_STEPS})
        resampled = resampled.rename(columns=rename)
        keep = ['DateTime', 'Date'] + list(rename.values())
        frames[seq] = resampled[keep]

    corridor = frames[ACTIVE_PORTALS[0]]
    for seq in ACTIVE_PORTALS[1:]:
        corridor = corridor.merge(frames[seq], on=['DateTime', 'Date'], how='inner')

    times = corridor.DateTime.dt.time
    am_counts = corridor.loc[(times >= AM_WINDOW_START) & (times <= AM_WINDOW_END)].groupby('Date').size()
    valid_days = set(am_counts[am_counts >= MIN_COVERAGE * expected_rows(AM_WINDOW_START, AM_WINDOW_END)].index)
    corridor = corridor[corridor.Date.isin(valid_days)].sort_values('DateTime').reset_index(drop=True)
    print(f'[Features] Evaluation corridor: {corridor.shape}; valid AM-window days={corridor.Date.nunique()}')
    return corridor


eval_corridor_df = build_eval_corridor(eval_raw_df)

# =============================================================================
# PREDICTIONS AND METRICS
# =============================================================================

def add_smoothed_actual(frame, target, window):
    raw = f'p{TARGET_PORTAL_SEQ}_{target}'
    name = f'{raw}_smoothed_w{window}'
    frame[name] = frame.groupby('Date')[raw].transform(
        lambda x: x.rolling(window=window, center=True, min_periods=1).mean()
    ) if window > 1 else frame[raw]
    return name


def make_prediction_rows(corridor, target, entry):
    features = entry['feature_cols']
    missing = [column for column in features if column not in corridor.columns]
    if missing:
        raise KeyError(f'Model feature mismatch for {target}; missing {missing}')

    smooth_window = int(entry.get('smooth_window', 1))
    delta_mode = bool(entry.get('predict_delta', False))
    actual_column = add_smoothed_actual(corridor, target, smooth_window)
    current_column = f'p{TARGET_PORTAL_SEQ}_{target}'
    rows = []

    for i in range(MAX_LAG, len(corridor) - FORECAST_STEPS):
        origin = corridor.loc[i, 'DateTime']
        if not (AM_WINDOW_START <= origin.time() <= AM_WINDOW_END):
            continue
        if corridor.loc[i, 'Date'] != corridor.loc[i + FORECAST_STEPS, 'Date']:
            continue

        x = corridor.loc[[i], features].to_numpy(dtype=float)
        scaled = entry['scaler'].transform(x)
        current = float(corridor.loc[i, current_column])
        predictions = np.asarray([entry['models'][step].predict(scaled)[0] for step in range(FORECAST_STEPS)])
        if delta_mode:
            predictions = predictions + current

        for step in range(FORECAST_STEPS):
            future_i = i + step + 1
            actual = float(corridor.loc[future_i, actual_column])
            predicted = float(predictions[step])
            residual = predicted - actual
            rows.append({
                'Target': target,
                'Origin_DateTime': origin,
                'Target_DateTime': corridor.loc[future_i, 'DateTime'],
                'Origin_Date': corridor.loc[i, 'Date'],
                'Origin_Time': origin.strftime('%H:%M'),
                'Origin_Day_Name': origin.day_name(),
                'Origin_Is_Weekend': int(origin.dayofweek >= 5),
                'Horizon_min': step + 1,
                'Current_Raw_Value': current,
                'Actual_Level': actual,
                'Predicted_Level': predicted,
                'Residual_Pred_minus_Actual': residual,
                'Absolute_Error': abs(residual),
                'Squared_Error': residual ** 2,
                'Target_Smoothing_Window': smooth_window,
                'Predict_Delta': delta_mode,
            })
    return pd.DataFrame(rows)


prediction_frames = []
for target in ACTIVE_TARGETS:
    print(f'[Predict] Producing all horizon predictions for target={target}...')
    prediction_frames.append(make_prediction_rows(eval_corridor_df, target, trained_models[target]))

prediction_detail_df = pd.concat(prediction_frames, ignore_index=True)
prediction_detail_df.to_csv(os.path.join(OUTPUT_DIR, 'evaluation_prediction_detail.csv'), index=False)
print(f'[Export] prediction detail: {len(prediction_detail_df):,} rows')


def calculate_metrics(frame):
    actual = frame.Actual_Level.to_numpy()
    predicted = frame.Predicted_Level.to_numpy()
    residual = predicted - actual
    absolute = np.abs(residual)
    return {
        'N': len(frame),
        'R2': r2_score(actual, predicted),
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'Bias_Mean_Error': residual.mean(),
        'Median_Error': np.median(residual),
        'Median_Absolute_Error': np.median(absolute),
        'P90_Absolute_Error': np.quantile(absolute, 0.90),
        'Max_Absolute_Error': absolute.max(),
    }

horizon_rows = []
for (target, horizon), group in prediction_detail_df.groupby(['Target', 'Horizon_min']):
    horizon_rows.append({'Target': target, 'Horizon_min': horizon, **calculate_metrics(group)})
horizon_metrics_df = pd.DataFrame(horizon_rows).sort_values(['Target', 'Horizon_min'])
horizon_metrics_df.iloc[:, 2:] = horizon_metrics_df.iloc[:, 2:].round(5)
horizon_metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'evaluation_horizon_metrics.csv'), index=False)

summary_rows = []
for target, group in prediction_detail_df.groupby('Target'):
    metrics = calculate_metrics(group)
    reliable = trained_models[target].get('documented_reliable_through_min', np.nan)
    summary_rows.append({'Target': target, 'Documented_Reliable_Through_Min': reliable, **metrics})
target_summary_df = pd.DataFrame(summary_rows)
target_summary_df.iloc[:, 2:] = target_summary_df.iloc[:, 2:].round(5)
target_summary_df.to_csv(os.path.join(OUTPUT_DIR, 'evaluation_target_summary.csv'), index=False)

print('\n' + '=' * 80)
print('EXTERNAL EVALUATION: PER-HORIZON METRICS')
print('=' * 80)
print(horizon_metrics_df.to_string(index=False))
print('\nOVERALL BY TARGET')
print(target_summary_df.to_string(index=False))

# =============================================================================
# SEGMENT, DAY, CALIBRATION, AND HIGH-ERROR DIAGNOSTICS
# =============================================================================

segment_rows = []
segment_specs = [
    ('Weekend_Status', 'Origin_Is_Weekend', {0: 'Weekday', 1: 'Weekend'}),
    ('Day_Of_Week', 'Origin_Day_Name', None),
    ('Origin_Time', 'Origin_Time', None),
]
for target, target_df in prediction_detail_df.groupby('Target'):
    for horizon, horizon_df in target_df.groupby('Horizon_min'):
        for segment_type, column, labels in segment_specs:
            for value, group in horizon_df.groupby(column):
                if len(group) < 3:
                    continue
                label = labels.get(value, value) if labels else value
                segment_rows.append({
                    'Target': target, 'Horizon_min': horizon,
                    'Segment_Type': segment_type, 'Segment': label,
                    **calculate_metrics(group),
                })
segment_metrics_df = pd.DataFrame(segment_rows)
if not segment_metrics_df.empty:
    numeric = segment_metrics_df.select_dtypes(include=np.number).columns
    segment_metrics_df[numeric] = segment_metrics_df[numeric].round(5)
segment_metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'evaluation_segment_metrics.csv'), index=False)

# Aggregation by origin date identifies consistently difficult days.
day_metrics_rows = []
for (target, date, horizon), group in prediction_detail_df.groupby(['Target', 'Origin_Date', 'Horizon_min']):
    day_metrics_rows.append({'Target': target, 'Origin_Date': date, 'Horizon_min': horizon, **calculate_metrics(group)})
day_metrics_df = pd.DataFrame(day_metrics_rows)
if not day_metrics_df.empty:
    numeric = day_metrics_df.select_dtypes(include=np.number).columns
    day_metrics_df[numeric] = day_metrics_df[numeric].round(5)
day_metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'evaluation_day_metrics.csv'), index=False)

# Worst origins across all horizon predictions.
origin_error = prediction_detail_df.groupby(['Target', 'Origin_DateTime'], as_index=False).agg(
    Mean_Absolute_Error=('Absolute_Error', 'mean'),
    Max_Absolute_Error=('Absolute_Error', 'max'),
    Mean_Bias=('Residual_Pred_minus_Actual', 'mean'),
    n_predictions=('Absolute_Error', 'size'),
).sort_values(['Target', 'Mean_Absolute_Error'], ascending=[True, False])
origin_error.to_csv(os.path.join(OUTPUT_DIR, 'evaluation_high_error_origins.csv'), index=False)

# Calibration: decile bins based on predicted level, per target/horizon.
calibration_rows = []
for (target, horizon), group in prediction_detail_df.groupby(['Target', 'Horizon_min']):
    if group.Predicted_Level.nunique() < 10:
        continue
    temp = group.copy()
    temp['Prediction_Bin'] = pd.qcut(temp.Predicted_Level, q=10, duplicates='drop')
    for bin_label, bin_df in temp.groupby('Prediction_Bin', observed=False):
        calibration_rows.append({
            'Target': target, 'Horizon_min': horizon, 'Prediction_Bin': str(bin_label),
            'N': len(bin_df), 'Predicted_Mean': bin_df.Predicted_Level.mean(),
            'Actual_Mean': bin_df.Actual_Level.mean(),
            'Calibration_Gap_Pred_minus_Actual': (bin_df.Predicted_Level - bin_df.Actual_Level).mean(),
        })
calibration_df = pd.DataFrame(calibration_rows)
if not calibration_df.empty:
    calibration_df.iloc[:, 3:] = calibration_df.iloc[:, 3:].round(5)
calibration_df.to_csv(os.path.join(OUTPUT_DIR, 'evaluation_calibration.csv'), index=False)

print('\n' + '=' * 80)
print('DIAGNOSTIC HIGHLIGHTS')
print('=' * 80)
for target in ACTIVE_TARGETS:
    subset = horizon_metrics_df[horizon_metrics_df.Target == target]
    neg = subset.loc[subset.R2 < 0, 'Horizon_min'].tolist()
    print(f"{target}: mean R2 over horizons={subset.R2.mean():.4f}; negative-R2 horizons={neg if neg else 'none'}")
    print(f"  Most difficult evaluation origins:\n{origin_error[origin_error.Target == target].head(5).to_string(index=False)}")

# =============================================================================
# PLOTS
# =============================================================================

COLORS = {'f': '#1f77b4', 'S_pred': '#d62728'}
LABELS = {'f': 'f (flow)', 'S_pred': 'S_pred (congestion index)'}

fig, ax = plt.subplots(figsize=(10, 5.5))
for target, group in horizon_metrics_df.groupby('Target'):
    ax.plot(group.Horizon_min, group.R2, marker='o', label=LABELS.get(target, target), color=COLORS.get(target))
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set(title='External Evaluation: R² by Forecast Horizon', xlabel='Horizon (minutes)', ylabel='R²')
ax.grid(alpha=0.3); ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'r2_by_horizon.png'), dpi=160); plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for target, group in horizon_metrics_df.groupby('Target'):
    axes[0].plot(group.Horizon_min, group.MAE, marker='o', label=LABELS.get(target, target), color=COLORS.get(target))
    axes[1].plot(group.Horizon_min, group.RMSE, marker='s', label=LABELS.get(target, target), color=COLORS.get(target))
for ax, metric in zip(axes, ['MAE', 'RMSE']):
    ax.set(title=f'External Evaluation: {metric}', xlabel='Horizon (minutes)', ylabel=metric)
    ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, 'mae_rmse_by_horizon.png'), dpi=160); plt.close(fig)

fig, axes = plt.subplots(1, len(ACTIVE_TARGETS), figsize=(6 * len(ACTIVE_TARGETS), 5), squeeze=False)
for ax, target in zip(axes[0], ACTIVE_TARGETS):
    subset = prediction_detail_df[(prediction_detail_df.Target == target) & (prediction_detail_df.Horizon_min == 1)]
    ax.scatter(subset.Actual_Level, subset.Predicted_Level, alpha=0.35, s=12, color=COLORS.get(target))
    low = min(subset.Actual_Level.min(), subset.Predicted_Level.min())
    high = max(subset.Actual_Level.max(), subset.Predicted_Level.max())
    ax.plot([low, high], [low, high], 'k--', linewidth=1)
    ax.set(title=f'{LABELS.get(target, target)}: +1 minute', xlabel='Actual level', ylabel='Predicted level')
    ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, 'predicted_vs_actual_scatter.png'), dpi=160); plt.close(fig)

fig, axes = plt.subplots(len(ACTIVE_TARGETS), 1, figsize=(11, 4.2 * len(ACTIVE_TARGETS)), squeeze=False)
for ax, target in zip(axes[:, 0], ACTIVE_TARGETS):
    subset = prediction_detail_df[prediction_detail_df.Target == target]
    data = [subset[subset.Horizon_min == h].Residual_Pred_minus_Actual.to_numpy() for h in range(1, FORECAST_STEPS + 1)]
    box = ax.boxplot(data, positions=range(1, FORECAST_STEPS + 1), patch_artist=True, showfliers=False)
    for patch in box['boxes']:
        patch.set_facecolor(COLORS.get(target)); patch.set_alpha(0.35)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.set(title=f'Residual Distribution by Horizon: {LABELS.get(target, target)}', xlabel='Horizon (minutes)', ylabel='Prediction − actual')
    ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, 'residual_by_horizon.png'), dpi=160); plt.close(fig)

print('\n' + '=' * 80)
print('EVALUATION COMPLETE')
print('=' * 80)
print(f'All CSV diagnostics and PNG plots saved in: {OUTPUT_DIR}')
print('For a genuinely external result, retain only evaluation dates never used in final model training or tuning.')

[Load] Forecaster: /content/drive/MyDrive/AH2179 GP5/prediction model v8.2 (portals 7-8, time-aware)/fS_prediction_model_p78_FINAL_cmax_v82.joblib
[Load] C_max v8.2: /content/drive/MyDrive/AH2179 GP5/prediction model v8.2 (portals 7-8, time-aware)/cmax_prediction_model_p78_v82.joblib
[Load] Portals=[7, 8]; target portal=8; targets=['f', 'S_pred']
  f: smooth_window=5, predict_delta=False, features=18
  S_pred: smooth_window=5, predict_delta=True, features=18
[Data warning] Using final_df. Confirm these dates were not used for final training.
[Data] Rows=2,456,648; dates=2021-06-01 04:00:00 to 2021-12-31 10:00:00; days=214
  [C_max] portal 7: 214 of 214 day(s) yielded C_max_pred.
  [C_max] portal 8: 214 of 214 day(s) yielded C_max_pred.
[Features] Evaluation corridor: (307081, 20); valid AM-window days=214
[Predict] Producing all horizon predictions for target=f...
[Predict] Producing all horizon predictions for target=S_pred...
[Export] prediction detail: 391,620 rows

EXTERNAL EVALUAT

### Plot

In [ ]:
"""
Interactive Part 2 Prediction Viewer -- FINAL C_max v8.2-Compatible Model
================================================================================
Interactive actual-vs-predicted plot for the final Part 2 model.

SELECTORS
---------
- Date: choose any available valid AM-window date
- Forecast instance: choose a minute between 07:30 and 08:30
- Target: f (flow) or S_pred (capacity-normalised congestion)

PLOT CONTENT
------------
- Grey dotted line: raw observed target values
- Black line: actual 5-minute-smoothed target values used by the final model
- Red dashed line / circles: model prediction at every +1 to +15 horizon
- Vertical dotted line: selected forecast-origin time (t=0)

Why two actual lines?
---------------------
The final model was deliberately trained to predict a five-minute centred
smoothed target, not the raw volatile one-minute value. The black line is
therefore the correct accuracy comparison. The grey line remains visible so
you can judge how much raw measurement/count noise is present.

REQUIRED SESSION STATE
----------------------
Preferred: run `v8_evaluate_final_cmax_v82.py` first in the SAME session.
It creates `eval_corridor_df`, reconstructed from holdout/evaluation data.

Alternative: run the final training script first in the SAME session; it
creates `corridor_df` and `horizon_models`.

MODEL LOADING
-------------
If `horizon_models` is not in memory, the viewer loads the exported final
artifact from Drive/output. It still needs either `eval_corridor_df` or
`corridor_df` in memory because this viewer intentionally does not rebuild
all C_max / S_pred features itself.

USAGE
-----
Run this script, then run in a new notebook cell:

    show_prediction_viewer()

Optional programmatic static plot:

    plot_prediction_instance(
        selected_date='2021-10-04',
        selected_time='07:47',
        target_name='f',
    )
"""

import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# =============================================================================
# CONFIGURATION
# =============================================================================

DRIVE_EXPORT_DIR = '/content/drive/MyDrive/AH2179 GP5/prediction model v8.2 (portals 7-8, time-aware)'
FINAL_MODEL_CANDIDATES = [
    os.path.join(DRIVE_EXPORT_DIR, 'fS_prediction_model_p78_FINAL_cmax_v82.joblib'),
    os.path.join('output', 'fS_prediction_model_p78_FINAL_cmax_v82.joblib'),
    'fS_prediction_model_p78_FINAL_cmax_v82.joblib',
]

AM_WINDOW_START = pd.Timestamp('1900-01-01 07:30').time()
AM_WINDOW_END = pd.Timestamp('1900-01-01 08:30').time()
CONTEXT_BEFORE_MINUTES = 10

TARGET_LABELS = {
    'f': 'f (flow)',
    'S_pred': 'S_pred (capacity-normalised congestion)',
}

# =============================================================================
# FIND MODEL AND FEATURE FRAME
# =============================================================================

def find_first_existing(paths):
    for path in paths:
        if os.path.isfile(path):
            return path
    return None


def load_models():
    if 'horizon_models' in globals():
        print('[Viewer] Using in-memory horizon_models.')
        return globals()['horizon_models'], {
            'forecast_steps': globals().get('FORECAST_STEPS', 15),
            'target_portal_seq': globals().get('TARGET_PORTAL_SEQ', 8),
            'active_forecast_targets': globals().get('ACTIVE_FORECAST_TARGETS', ['f', 'S_pred']),
        }

    model_path = find_first_existing(FINAL_MODEL_CANDIDATES)
    if model_path is None:
        raise FileNotFoundError(
            'Final Part 2 artifact not found. Checked:\n  ' + '\n  '.join(FINAL_MODEL_CANDIDATES)
        )
    artifact = joblib.load(model_path)
    print(f'[Viewer] Loaded final artifact: {model_path}')
    return artifact['trained_models'], artifact


def load_feature_frame():
    if 'eval_corridor_df' in globals():
        print('[Viewer] Using eval_corridor_df (recommended: evaluation/holdout feature frame).')
        return globals()['eval_corridor_df'].copy(), 'evaluation data'
    if 'corridor_df' in globals():
        print('[Viewer warning] Using corridor_df from model training. This is useful for visual inspection but not external validation.')
        return globals()['corridor_df'].copy(), 'training data'
    raise RuntimeError(
        'No feature frame found. Run the evaluation script first to create eval_corridor_df, '
        'or run the final training script first to create corridor_df.'
    )


horizon_models, model_metadata = load_models()
viewer_df, viewer_data_source = load_feature_frame()

FORECAST_STEPS = int(model_metadata.get('forecast_steps', 15))
TARGET_PORTAL_SEQ = int(model_metadata.get('target_portal_seq', 8))
AVAILABLE_TARGETS = [
    target for target in model_metadata.get('active_forecast_targets', horizon_models.keys())
    if target in horizon_models
]

viewer_df['DateTime'] = pd.to_datetime(viewer_df['DateTime'])
viewer_df = viewer_df.sort_values('DateTime').reset_index(drop=True)
viewer_df['_Time'] = viewer_df['DateTime'].dt.time
viewer_df['_DateString'] = viewer_df['DateTime'].dt.strftime('%Y-%m-%d')

am_mask = (viewer_df['_Time'] >= AM_WINDOW_START) & (viewer_df['_Time'] <= AM_WINDOW_END)
available_dates = sorted(viewer_df.loc[am_mask, '_DateString'].unique())
if not available_dates:
    raise ValueError('No 07:30-08:30 AM-window instances exist in the available feature frame.')

print(f'[Viewer] Ready: {len(available_dates)} selectable dates; targets={AVAILABLE_TARGETS}; source={viewer_data_source}.')

# =============================================================================
# DATA / PREDICTION HELPERS
# =============================================================================

def smoothed_actual_column(target_name, entry):
    """Constructs the same per-day centred rolling target used in final model
    training/evaluation. This is the correct actual series for model scoring."""
    raw_column = f'p{TARGET_PORTAL_SEQ}_{target_name}'
    window = int(entry.get('smooth_window', 1))
    column_name = f'_viewer_{raw_column}_smoothed_w{window}'
    if column_name not in viewer_df.columns:
        if window <= 1:
            viewer_df[column_name] = viewer_df[raw_column]
        else:
            viewer_df[column_name] = viewer_df.groupby('Date')[raw_column].transform(
                lambda values: values.rolling(window=window, center=True, min_periods=1).mean()
            )
    return raw_column, column_name


def valid_instances_for_date(date_string):
    subset = viewer_df[(viewer_df['_DateString'] == date_string) & am_mask].copy()
    # A forecast origin needs all +1..+15 future points inside the same day.
    valid_times = []
    for index in subset.index:
        end_index = index + FORECAST_STEPS
        if end_index >= len(viewer_df):
            continue
        if viewer_df.loc[index, 'Date'] != viewer_df.loc[end_index, 'Date']:
            continue
        valid_times.append(viewer_df.loc[index, 'DateTime'].strftime('%H:%M'))
    return valid_times


def locate_instance(date_string, time_string):
    matches = viewer_df[
        (viewer_df['_DateString'] == date_string) &
        (viewer_df['DateTime'].dt.strftime('%H:%M') == time_string)
    ]
    if matches.empty:
        raise ValueError(f'No row found for date={date_string}, time={time_string}.')
    return int(matches.index[0])


def predict_all_horizons(row_index, target_name):
    entry = horizon_models[target_name]
    features = entry['feature_cols']
    x = viewer_df.loc[[row_index], features].to_numpy(dtype=float)
    scaled_x = entry['scaler'].transform(x)
    predictions = np.asarray([
        entry['models'][step].predict(scaled_x)[0]
        for step in range(FORECAST_STEPS)
    ])

    raw_column = f'p{TARGET_PORTAL_SEQ}_{target_name}'
    current_value = float(viewer_df.loc[row_index, raw_column])
    if entry.get('predict_delta', False):
        predictions = predictions + current_value
    return predictions, current_value

# =============================================================================
# PLOT FUNCTION
# =============================================================================

def plot_prediction_instance(selected_date, selected_time, target_name):
    """Draws one actual-versus-predicted forecast fan and prints a precise
    horizon comparison table below it."""
    if target_name not in AVAILABLE_TARGETS:
        raise ValueError(f'Unknown target {target_name}; choices: {AVAILABLE_TARGETS}')

    row_index = locate_instance(selected_date, selected_time)
    if row_index + FORECAST_STEPS >= len(viewer_df):
        raise ValueError('Selected instance does not have a complete 15-minute future horizon.')
    if viewer_df.loc[row_index, 'Date'] != viewer_df.loc[row_index + FORECAST_STEPS, 'Date']:
        raise ValueError('Selected instance crosses a calendar-day boundary before +15 minutes.')

    entry = horizon_models[target_name]
    raw_column, smooth_column = smoothed_actual_column(target_name, entry)
    predictions, current_value = predict_all_horizons(row_index, target_name)

    start_index = max(0, row_index - CONTEXT_BEFORE_MINUTES)
    end_index = min(len(viewer_df) - 1, row_index + FORECAST_STEPS)
    view = viewer_df.loc[start_index:end_index, ['DateTime', raw_column, smooth_column]].copy()
    view['minutes_relative'] = (
        view['DateTime'] - viewer_df.loc[row_index, 'DateTime']
    ).dt.total_seconds() / 60.0

    horizons = np.arange(1, FORECAST_STEPS + 1)
    future_rows = viewer_df.loc[row_index + 1:row_index + FORECAST_STEPS]
    actual_raw = future_rows[raw_column].to_numpy(dtype=float)
    actual_smooth = future_rows[smooth_column].to_numpy(dtype=float)
    errors = predictions - actual_smooth

    origin_time = viewer_df.loc[row_index, 'DateTime']
    reliable_until = entry.get('documented_reliable_through_min', FORECAST_STEPS)
    target_label = TARGET_LABELS.get(target_name, target_name)

    fig, ax = plt.subplots(figsize=(12, 6.5))
    ax.plot(
        view['minutes_relative'], view[raw_column],
        color='#9E9E9E', linestyle=':', linewidth=1.7, marker='o', markersize=3,
        label='Raw observed value'
    )
    ax.plot(
        view['minutes_relative'], view[smooth_column],
        color='black', linewidth=2.2, marker='o', markersize=3.5,
        label=f'Actual model target (smoothed, w={entry.get("smooth_window", 1)})'
    )
    ax.plot(
        horizons, predictions,
        color='#D62728', linestyle='--', linewidth=2.0, marker='o', markersize=5,
        label='Predicted level (+1 to +15 min)'
    )
    ax.axvline(0, color='#444444', linestyle=':', linewidth=1.5, label='Forecast origin (t=0)')
    ax.axvspan(reliable_until + 0.5, FORECAST_STEPS + 0.5, color='#F4A261', alpha=0.12,
               label=f'Informational range after +{reliable_until} min')

    ax.set_title(
        f'Portal {TARGET_PORTAL_SEQ}: {target_label}\n'
        f'Forecast origin: {origin_time:%Y-%m-%d %H:%M} | '
        f'source: {viewer_data_source}'
    )
    ax.set_xlabel('Minutes relative to selected forecast origin')
    ax.set_ylabel(target_label)
    ax.grid(alpha=0.25)
    ax.legend(loc='best')
    fig.tight_layout()
    plt.show()

    comparison = pd.DataFrame({
        'Horizon_min': horizons,
        'Predicted_Level': predictions,
        'Actual_Smoothed_Level': actual_smooth,
        'Actual_Raw_Level': actual_raw,
        'Residual_Pred_minus_Smoothed_Actual': errors,
        'Absolute_Error_vs_Smoothed_Actual': np.abs(errors),
        'Within_Documented_Deployment_Range': horizons <= reliable_until,
    })

    print('\nSelected origin details')
    print(f'  Date/time: {origin_time}')
    print(f'  Target: {target_name}')
    print(f'  Current raw value at t=0: {current_value:.6f}')
    print(f'  Smoothing window: {entry.get("smooth_window", 1)} minute(s)')
    print(f'  Delta prediction: {entry.get("predict_delta", False)}')
    print(f'  Documented deployment range: +1 to +{reliable_until} minute(s)')
    print('\nHorizon comparison:')
    print(comparison.round(5).to_string(index=False))

    return comparison

# =============================================================================
# INTERACTIVE WIDGET
# =============================================================================

def show_prediction_viewer():
    """Displays linked Date / Instance / Target dropdown selectors."""
    date_dropdown = widgets.Dropdown(
        options=available_dates,
        value=available_dates[0],
        description='Date:',
        layout=widgets.Layout(width='350px'),
    )
    initial_times = valid_instances_for_date(available_dates[0])
    if not initial_times:
        raise ValueError(f'No usable full-horizon instances on {available_dates[0]}.')
    time_dropdown = widgets.Dropdown(
        options=initial_times,
        value=initial_times[0],
        description='Instance:',
        layout=widgets.Layout(width='350px'),
    )
    target_dropdown = widgets.Dropdown(
        options=AVAILABLE_TARGETS,
        value=AVAILABLE_TARGETS[0],
        description='Target:',
        layout=widgets.Layout(width='350px'),
    )

    def update_time_options(change):
        times = valid_instances_for_date(change['new'])
        time_dropdown.options = times
        if times:
            time_dropdown.value = times[0]

    date_dropdown.observe(update_time_options, names='value')

    output = widgets.interactive_output(
        plot_prediction_instance,
        {
            'selected_date': date_dropdown,
            'selected_time': time_dropdown,
            'target_name': target_dropdown,
        },
    )

    selector_box = widgets.VBox([
        widgets.HTML(
            '<b>Part 2 Final Viewer</b><br>'
            'Grey = raw observed value; black = smoothed actual target; red = forecast. '
            'The shaded right-side region is beyond the target\'s documented deployment horizon.'
        ),
        date_dropdown,
        time_dropdown,
        target_dropdown,
    ])
    display(selector_box, output)


if __name__ == '__main__':
    print('\n[Viewer] Initialisation complete.')
    print('[Viewer] Run: show_prediction_viewer()')
    show_prediction_viewer()

[Viewer] Using in-memory horizon_models.
[Viewer] Using eval_corridor_df (recommended: evaluation/holdout feature frame).
[Viewer] Ready: 214 selectable dates; targets=['f', 'S_pred']; source=evaluation data.

[Viewer] Initialisation complete.
[Viewer] Run: show_prediction_viewer()


Output()